# Gemma Agent Trajectory SFT

**公开工程复盘与CPU教学示例 / Public engineering notes and synthetic examples**

[GitHub仓库](https://github.com/fjnuslw/gemma-agent-trajectory-sft) · [官方竞赛](https://www.kaggle.com/competitions/gemma-4-developer-agent)

证据截点2026-11-16。历史C01/C02完成训练与短加载；C03仅有23/124步的用户进度，96题次检查点面板待执行/待回传。没有稳定LoRA净收益或隐藏泛化结论。

本Notebook包含与GitHub同步的技术说明、42项证据主张、脱敏源摘录和四个新写的标准库教学代码单元。代码输入为合成数据，不是官方Gemma token、真实轨迹或模型输出；真实任务/训练指标均单独标明来源。平台发布采用Quick Save，本次不启动计算或竞赛评分提交。

原创说明与教学代码采用MIT；第三方内容保留原有权利和来源，见最后的NOTICE。

## 先用合成例子理解四个合同
以下四个单元可在CPU逐一执行，只演算定义；它们不替代原生模板检查、31B训练或官方任务判分。

In [ ]:
# Synthetic task/stage weighting; not a model experiment.
from fractions import Fraction
counts = {'task_a': {'identify': 2, 'edit': 1, 'verify': 1, 'recover_finish': 1},
          'task_b': {'identify': 1, 'edit': 2, 'verify': 1}}
N = sum(sum(stages.values()) for stages in counts.values())
T = len(counts)
mass = {}
weighted_rows = []
for task, stages in counts.items():
    mass[task] = Fraction(0)
    for stage, n in stages.items():
        m = Fraction(N, T * len(stages) * n)
        weighted_rows.extend([m] * n)
        mass[task] += n * m / N
assert sum(weighted_rows) / N == 1
assert set(mass.values()) == {Fraction(1, T)}
print('N:', N, 'mean multiplier:', float(sum(weighted_rows) / N))
print('task mass:', {k: float(v) for k, v in mass.items()})


In [ ]:
# Teaching positions only: this is NOT the actual Gemma template/tokenizer.
tokens = ['user', 'tool_receipt', 'assistant_start', 'explanation', 'tool_call', 'handoff']
prompt_length = 3
labels = [-100] * prompt_length + list(range(prompt_length, len(tokens)))
reason_positions = {3}
action = [i if labels[i] != -100 and i not in reason_positions else -100 for i in range(len(tokens))]
reason = [i if i in reason_positions else -100 for i in range(len(tokens))]
assert all(labels[i] == -100 for i in range(prompt_length))
assert all(not (a != -100 and r != -100) for a, r in zip(action, reason))
assert all((a != -100 or r != -100) == (y != -100) for a, r, y in zip(action, reason, labels))
print(list(zip(tokens, labels, action, reason)))


In [ ]:
# Tiny mathematical CE illustration; no model, PyTorch or GPU.
import math
logits = [[.2, .7, -.1], [1., -.2, .4], [.1, .3, .8]]
targets = [-100, 2, 1]
def ce(row, target):
    maximum = max(row)
    return math.log(sum(math.exp(v - maximum) for v in row)) + maximum - row[target]
selected = [i for i, target in enumerate(targets) if target != -100]
dense_masked_mean = sum(ce(row, y) for row, y in zip(logits, targets) if y != -100) / len(selected)
selected_mean = sum(ce(logits[i], targets[i]) for i in selected) / len(selected)
assert abs(dense_masked_mean - selected_mean) < 1e-12
print('masked dense CE = selected CE:', selected_mean)


In [ ]:
# Synthetic key-count demonstration; no adapter weights are loaded.
global_layers = set(range(5, 60, 6))
native = set()
for layer in range(60):
    projections = ['q', 'k', 'o', 'gate', 'up', 'down'] + ([] if layer in global_layers else ['v'])
    native.update((layer, projection, ab) for projection in projections for ab in ['A', 'B'])
serving = native | {(layer, 'v', ab) for layer in global_layers for ab in ['A', 'B']}
assert len(native) == 820 and len(serving) == 840 and len(serving - native) == 20
print('native:', len(native), 'serving:', len(serving), 'K-to-V copies:', len(serving - native))


# Gemma Agent Trajectory SFT

**可执行轨迹、可信训练目标与长序列LoRA的工程复盘**  
Documentation and synthetic CPU examples for an evidence-grounded coding-agent SFT project.

[Kaggle同版Notebook](https://www.kaggle.com/code/fjnulinweisong/gemma-agent-trajectory-sft) · [技术总览](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/technical-overview.zh-CN.md) · [证据与指标](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/evidence-and-metrics.zh-CN.md) · [技术追问](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/interview-guide.zh-CN.md) · [发布范围](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/release-notes.zh-CN.md)

本项目围绕公开仓库issue修复，接入官方SDK、ADK工具与独立判分，将原始HTTP、实际工具动作和回执关联，再把有资格的学生前缀接续整理成原生SFT目标。训练侧组合冻结BF16底座、FP32 LoRA、选择性FSDP2、长序列分块重算及共享K/V服务导出。

本仓库是**文档与教学示例发行**。完整竞赛执行器、真实轨迹/数据和模型权重不在附件中。源证据使用带原行号的脱敏有限摘录；所有“已验证”指标归属于相应既有报告，不是本Notebook新运行的模型结果。

## 四个可解释的技术点

- **执行证据**：HTTP → ADK实际调用/回执 → patch → 独立verification，区分文本伪工具与真实执行。
- **训练资格**：真实学生请求边界、受控教师接续、公开短说明、严格原生token mask和题/阶段均权。
- **长序列训练**：冻结decoder由FSDP2分片，LoRA复制并显式同步，处理共享head、非浮点辅助参数和尾批通信。
- **交付兼容**：native820→serving840的共享K/V适配；精确重复引用回传保留行为证据。

```mermaid
flowchart LR
    A[官方任务执行] --> B[HTTP / ADK / 工具回执]
    A --> J[patch与独立判分]
    B --> C[真实学生前缀与受控接续]
    C --> D[资格审选 / 原生mask / 均权]
    D --> E[四卡冻结底座LoRA训练]
    E --> F[820到840服务导出与短加载]
    F -. C03待回传 .-> G[固定checkpoint开发对照]
```

## 规模与结果边界

| 项目 | 有据可查的事实 |
|---|---|
| C03数据 | 990目标回合 / 160任务ID / 199,458正token，最长26,910，无截断 |
| LoRA | 60文本层 / 410投影 / 122,429,440训练参数，标准r16 |
| 历史C02训练 | 524更新、4,180次真实行消费；最长26,979两累积，真实服务短加载通过 |
| C01同温度开发对照 | base T.2=51/129；C01 T.2=48/129；同LoRA T1=52/129但缺匹配base T1 |
| 回传瘦身实例 | 1,706,923,496B→154,622,101B，减少90.94%，HTTP/ADK行为证据保留 |
| 当前未完成 | C03只接到23/124步用户进度；96题次面板待执行/待回传；外部留出与精确resume待完成 |

**没有建立LoRA稳定任务净收益或隐藏泛化结论。** 协议指标改善、训练loss下降、服务加载成功和整题通过是不同证据。C02只有主动中断的部分行为窗口，不能补成129成绩。本项目没有执行GRPO、DPO或在线RL。

## 阅读与本地教学演算

1. 从[技术总览](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/technical-overview.zh-CN.md)了解架构与取舍。
2. 用[42项证据主张](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/evidence-and-metrics.zh-CN.md)、[脱敏摘录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md)和[机器指标](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/evidence/metrics.json)检查事实边界。
3. 打开[Notebook](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/notebooks/gemma-agent-trajectory-sft.ipynb)，四个代码单元只用Python标准库演算合成例子，可在本地CPU逐单元运行。

Kaggle发布采用Quick Save，不启动平台计算；Notebook中的代码没有模型、网络请求或训练入口。完整GPU实验不属于此教学复现范围。

## 来源与许可

官方SDK/ADK、Gemma、PyTorch/FSDP2、PEFT和vLLM均归属各自上游。部分内存设计参考公开训练kit，具体来源见[NOTICE](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/NOTICE.md)。原创说明与新增教学代码采用[MIT](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/LICENSE)；第三方引用、模型和数据仍遵守各自条款。

证据整理截点：2026-11-16。后续训练与面板结果只在取得实际回传后更新。


# Gemma 编码 Agent 与轨迹 SFT：技术总览

> 范围：公开技术说明与教学资料。核对日期：2026-11-16。本文依据当前源码及既有执行回传整理，没有新运行模型、训练、评测或教师采集。源证据使用脱敏摘录；本仓库不提供完整竞赛执行环境。

## 1. 项目解决的问题与当前完成度

项目围绕软件仓库 issue 修复，搭建了“官方任务执行与独立判分 → 轨迹取证与错误诊断 → 真实学生状态上的受控教师接续 → 合格目标的原生模板 SFT → 服务导出与行为对照”的工程链路。重点是让模型产生可执行、可验证的源码修改，并把失败变成有来源、有资格的训练目标。

可证实的技术价值集中在三处：保持官方执行与判分语义的可观测运行框架；逐动作资格和逐 token 边界约束的数据生产；在四张 A800 上承载 Gemma 31B 冻结底座、完整长序列的 LoRA 训练与服务兼容。官方 SDK、ADK、vLLM、PEFT、FSDP2 和模型本身均为上游能力，项目贡献是适配、组合、边界处理和验证。

| 组成 | 本次可确认状态 | 不能据此声称 |
|---|---|---|
| 官方任务执行、独立判分、HTTP/ADK 取证 | 历史完整双臂 258 题臂有回传；实际 6,464 动作与 7,031 请求已核绑定 | 隐藏评测环境与本地完全相同 |
| C01/C02 四卡 LoRA 训练、820→840 导出与真实短加载 | 已有目标机训练回传和独立接纳 | 训练成功即任务提分 |
| C03 数据与训练链路 | 990 目标回合、160 任务 ID 已冻结；CPU 合同及发行核验完成 | C03 已完成训练或已达到某线上分数 |
| C03 训练进度 | 用户报告 23/124 步、loss 1.25→0.35；原始训练回传尚未接纳 | 将用户进度转述写成已独立验证的训练曲线 |
| C03 检查点×温度面板 | base/50%/75%/100% × T0.2/T1.0 × 12 题，共 96 题次；交付完成，执行尚未确认 | 96 题次已经运行或已选出最佳 checkpoint |

最新状态取自 [C03 动态规格](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s001)、[进度记录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s002)及[面板报告](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s003)。早期 README、研究报告中的 NOT_RUN 是当时快照；不能将不同时间的段落拼成当前完成状态。

## 2. 系统架构

```mermaid
flowchart TB
    Task[公开 issue 与冻结仓库快照] --> Eval[官方 Evaluator / ADK 单 Agent 循环]
    Policy[版本化 YAML / 提示 / 六工具 / 采样与预算] --> Eval
    Eval <--> Serve[vLLM：官方 QAT 底座 / 指定 LoRA 路由]
    Eval <--> Tools[官方原生工具 / 任务工作区]
    Eval --> Patch[最终 patch]
    Patch --> Judge[独立验证工作区与官方判分]
    Judge --> Score[TaskResult 与逐题结果]
    Eval --> Trace[原始 HTTP / ADK / 工具回执 / 工作区证据]
    Trace --> Review[错误分类与真实请求切点]
    Review --> Prefix[重建学生前缀与任务状态]
    Prefix --> Teacher[盲教师：公开短说明与下一动作]
    Teacher --> Replay[官方 CPU 重放与实际工具执行]
    Replay --> Qualify[逐动作语义审选 / 来源与执行绑定]
    External[公开 Native / A 来源及仓库 split] --> Qualify
    Qualify --> Native[真实模板 / tokenizer / loss mask / 阶段权重]
    Native --> Freeze[不可变数据快照与训练输入合同]
    Freeze --> Train[四卡 FSDP2 / 冻结 BF16 底座 / FP32 LoRA]
    Train --> Export[native 820 张量 → serving 840 张量]
    Export --> Load[SDK 编译 / 数值差异短加载]
    Load -. 已准备，待 C03 实跑 .-> Panel[检查点与温度开发面板]
    Panel -. 待选择与明确执行 .-> Eval
```

图中的教师链属于离线数据生产；正式学生评测不接入教师。判分材料可供离线诊断与资格核实，不能作为下一动作教师的后验答案输入。局部动作有效、整题辅助通过、学生独立通过分别记录。

### 主要模块与技术栈

| 模块 | 主要位置 | 职责与依赖 |
|---|---|---|
| 学生运行 | `runtime/baseline.py`、`runtime/submission.py` | 调用官方 Evaluator、模型 registry 和 sandbox；物化版本化提交 |
| 取证 | `runtime/request_log.py`、`runtime/evidence.py`、`runtime/trace_audit.py` | 被动 HTTP 转发记录、ADK 事件与补丁/验证绑定；本地诊断组件 |
| 学习入口 | `learning_pipeline/__main__.py`、`core.py` | 轮次、审查、作业、数据、训练与评测交接的统一 CLI |
| 教师接续 | `collection/runner.py`、`capture_stage.py`、`equivalence.py` | 请求边界重建、状态比对、盲教师、官方重放及执行绑定 |
| 数据 | `dataset.py`、`composition.py`、`fresh.py`、`native.py` | 资格导入、来源混合、显式派生、原生渲染、mask、均权和审计 |
| 训练 | `training/train_adapter.py`、`train_core.py`、`fsdp_runtime.py` | Transformers / PEFT / PyTorch；冻结底座、长序列 loss、分片与梯度同步 |
| 快照 | `training/schedule.py`、`checkpoint.py` | optimizer-step 调度、native adapter 保存回读、最终状态留存 |
| 服务与交付 | `server/`、`evaluation/`、`templates/stage_compact.py` | 训练输入合同、K/V 导出、SDK/加载检查、安全展开、失败与回传处理 |

已冻结的目标训练栈为 PyTorch 2.10.0+cu128、Transformers 5.13.1、PEFT 0.21.2、Accelerate 1.15.0、compressed-tensors 0.15.0.1、Tokenizers 0.22.2、Safetensors 0.8.0、Jinja2 3.1.6。评测使用 swegemma 0.2.11、adk-submission 0.2.13、adk-eval-core 0.1.0、google-adk 1.36.1 及比赛 vLLM 0.19.1 构建；同一版本号不替代具体 wheel/源码绑定。本地微型测试涉及 Torch 2.14 CPU，与目标 CUDA 栈分开记载。[训练依赖](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s004) · [评测依赖](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s005)

## 3. 官方任务执行、上下文与取证

### 3.1 让模型解题，独立环境判分

当前研究对象采用一个 `LlmAgent`，通过官方工具读写源码和运行检查，产出 patch 后交独立验证。六工具为 `get_status`、`submit_patch`、`read_file`、`run_command`、`edit_file`、`write_file`；这是本项目选定的接口集合，官方还有其他原生工具，并非平台只有六个。

r04/C03 的冻结口径为每题 480 秒、28 次计费调用、80 轮；命令超时 240 秒，独立验证 300 秒。status/submit 的免费调用应与计费调用分开。采样 top_p=.95、top_k=64，总输出上限 8192、thinking_budget=4096。模型的自然语言提示不能代替官方硬预算；输出预算也不是训练上下文长度。相关参数以[冻结轮次配置](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s006)和实际输入 ZIP 为准，不能拿当前活动配置解释旧运行。

本地 runner 直接构造官方 `Evaluator(EvalConfig(...))`，保留官方工具、ADK 主循环、预算与独立判分；观测层保存 Agent 工作区证据，不读验证工作区来替模型修复。[运行接线](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s007) · [逐题结果与 patch 保存](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s008)

### 3.2 上下文压缩是官方能力，项目核实它怎样生效

配置使用 ADK `EventsCompactionConfig`：interval=15、overlap=2、token_threshold=14336、event_retention_size=5，另有 context cache 配置。项目没有发明另一套在线摘要 Agent 循环；工作是固定配置、捕获摘要请求及后续主请求，检查摘要是否真正送达、哪些原始证据被替代、摘要有没有失真。

C01 双臂报告核到 167/141 个已返回可解析摘要全部进入后续主请求；这能排除该批“摘要普遍漏发”的解释，不能证明摘要始终准确或压缩没有损害。C02 部分评测的 39 个摘要均送达，但存在编辑状态判断失真。判断压缩问题要同时看送达与内容正确性。[压缩接线](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s009) · [C01 观察](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s010) · [C02 观察](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s011)

### 3.3 把“生成了什么”与“执行了什么”关联起来

HTTP recorder 是本地被动 relay，请求 body 原样转发，保存 request、response.raw、metadata；raw 是去除 HTTP 传输 framing 后的实体字节，可包含 SSE，不是模型内部 token 流。取消、失败和未终结响应保留其状态，不补造 usage。ADK 事件记录实际 function call 和 function response，离线 audit 再把 HTTP 参数、ADK、官方 trace、patch 与验证结果关联。

C01 双臂已有 7,031 HTTP 请求与 6,464 工具动作的绑定审查。这个数量是相应运行的审计覆盖，不是并发量、用户量或服务 QPS，也不是全部 reasoning 逐字语义审阅。记录器与离线审计不随评分 ZIP 上线。[HTTP 记录器](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s012) · [证据层边界](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s013) · [真实覆盖](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s014)

### 3.4 先校准评测器，再比较 Agent

项目保留公开 129 题的空补丁/参考补丁控制证据，两个控制条件合计 258 题次。既有视图是 117 题参考可过，其中 2 题空补丁也可过，故 115 题可判别；其余 12 题为已知校准限制。129=115+2+12，不能将控制题或环境限制悄悄从原始官方结果中删除。

新面板复用全量校准绑定，只执行从中选出的 12 个开发题；“完整校准证据已绑定”和“新执行范围为 12 题”分别验证。参考补丁控制证明本地某些环境和裁判条件可用，不能证明隐藏镜像完全相同。[校准口径](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s015) · [面板校准与运行范围分离](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s016)

## 4. 从失败轨迹到可训练目标

### 4.1 错误分类服务于训练设计

项目把失败拆成调用/编辑表示、行动推进与恢复时机、实现语义与覆盖等类别。标签允许重叠；“已定位未修改”“检查没有击中真实入口”“JSON 合法却执行了破坏性命令”“修复遗漏兼容分支”需要不同目标，不能全部归为工具格式错误。

选择切点时绑定实际学生请求、来源臂、工具回执和当时状态。看到另一温度臂成功，不意味着可以把它的最终 patch 嫁接成当前学生的真实下一步。分析者提出的 rewrite 建议只作为诊断 metadata，不能当成执行前教师输出。[分类边界](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s017) · [逐题复盘](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s018)

### 4.2 受控教师短接续：先有真实状态，再有新动作

采集按边界逐步重建：从原仓库快照重放已冻结学生动作，捕获官方下一请求与任务文件状态；干净教师只见该请求，生成公开短说明和下一工具动作；再启动独立官方重放，消费该动作并保存真正的工具回执。`target_binding` 是计划，只有 `execution_binding.status=EXECUTED` 且请求/状态比对通过、唯一 ADK 调用与回执对应，才证明实际执行。

教师等待发生在已关闭重放之外，因此离线教师时间、真实重放耗时和原学生时钟分别记录。这不是暂停官方比赛时钟，也不是严格在线 on-policy RL。首次工作区没有原现场观测时标记 RECONSTRUCTED；请求比对允许的路径、时钟和少量展示顺序差异有明确规则，不能把声明等价说成全字节相同。[执行绑定](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s019) · [实际教师输入](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s020) · [等价范围](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s021)

短片段只证明局部动作或有效诊断；没有 TaskResult 时，整体验证为 NOT_RUN。自然闭合链才可记录真正的 resolved，并且官方通过仍不能替代逐动作语义审选。普通安全只读可由主审与机械证据支持；源码写入、验证结论、正说明和边界访问需要明确审核来源。[短片段资格](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s022) · [自然闭合资格](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s023)

### 4.3 公开说明、模型可见 reasoning 与私人 thinking

教师的 `decision_summary` 是专门撰写的公开训练内容，描述已观察证据及下一动作理由，产生于实际动作执行前；元数据标 `TEACHER_PUBLIC_DECISION_SUMMARY`。项目不请求或记录教师私人思维链。学生响应和公开数据中已有可见说明，也必须按真实来源、整段正确性和已读范围决定资格。

未全文审阅的 reasoning 保持 UNKNOWN/masked，不因为动作正确就自动变成正例。C03 完整 completion 视图中，未合格的“当前说明”会显式移除后重新渲染，避免它虽然不计 loss，却留在输入里泄露当前答案；历史说明的来源处理另外记录。[教师协议](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s024) · [完整 completion 约束](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s025)

### 4.4 用真实模板和 tokenizer 确定监督边界

一个训练行是“完整真实请求前缀＋一个合格的当前助手回合”，不是一整道题。相邻行可以共享历史，但历史、旧助手回合和工具回执全部 label=-100；只监督本行当前目标。完整输入仍参与因果前向，mask 不代表删除上下文或切断上下文的梯度路径。

`Native` 从冻结 `chat_template.jinja` 和 `tokenizer.json` 渲染 prompt/full，先核文本前缀，再核 token ID 严格前缀，并用字符跨度和 tokenizer offset 确定目标。C03 `complete_example` 要求全部实际工具调用均有资格；action 与说明标签不重叠，并集精确覆盖当前 completion，包含真实工具标记和 handoff。超长行隔离，无静默截断。[原生前缀检查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s026) · [完整目标与分量 mask](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s027)

实际处理过的边界问题是公开 A 转换来源在工具回执后预置了 18 字符 thought 入口，而当前模板将其放入 completion。修复按源历史的明确分支重建，仍逐字核原 prefix、completion、IDs 和 token 数；没有通过忽略差异放宽准入。全部 A14 条轨迹的 382 个源目标机械重建通过，其中选中 37 个；这不是 382 个新训练目标，也不是本 SDK 重执行。[修复及边界证据](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s028)

### 4.5 当前 C03 数据：规模、暴露与阶段均权

| 来源 | 目标回合 | 任务 ID | 证据边界 |
|---|---:|---:|---|
| 本地 D04 合格目标重选 | 720 | 113 | 复用已有资格；不是原 2090 行再完整训练 |
| 公开 Native | 233 | 41 | 原生捕获来源支持；本轮未在本 SDK 重执行 |
| 公开 A | 37 | 6 | 转换来源支持；严格重建源边界 |
| 合计 | 990 | 160 | 目标回合数不等于独立任务数 |

外部训练部分为 47 个任务 ID、45 个仓库；dev/final 各留出 8 仓库、9 issue，未入训练，环境尚未重建。本地 129 题早已用于开发和训练，不能称盲测。外部审选阅读 85 条轨迹、1,532 个可见回合，205 段保留的非空说明全文核准；这不是整个公开母集全部语义审核。

冻结数据共有 12,199,343 个输入位置、199,458 个正 token，其中行动/边界 156,980、合格说明 42,478；721 行有说明，7 行为事实 final。最长 26,910，上限 32,768；最大当前 completion 为 2,208，无截断。本轮没有新增教师调用。[最终统计与审查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s029)

四阶段为定位 313、编辑 271、验证 195、恢复收尾 211。每题只对实际存在的阶段分配质量，阶段内再分给目标回合：

\[
p_{i,s,j}=\frac{1}{T K_i n_{i,s}},\qquad m_{i,s,j}=N p_{i,s,j}.
\]

其中 T 是任务数，K_i 是题 i 实际保留的阶段数，n_i,s 是该题该阶段目标数。均匀、无放回遍历时用 m 乘行内 CE，平均倍率为 1；不能同时按 p 重采样后再乘同一权重。实际倍率 0.1546875–3.09375。这避免长轨迹或双温度来源自然获得更多任务质量，但阶段分类和配比本身仍是待行为验证的设计。[权重实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s030)

## 5. 训练工程与服务兼容

### 5.1 从官方 QAT 权重恢复冻结 BF16，只更新 LoRA

训练从原 compressed-tensors QAT 包恢复已存在的权重值：在固定 Transformers 版本使用 `run_compressed=False`，以 BF16 加载，检查 missing/unexpected/mismatched/error，再关闭已恢复模块的 QDQ 包装并冻结底座。没有重新做 NF4 量化，不是 31B 全参训练，也不是重新预训练。[底座加载与冻结](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s031)

LoRA 通过 PEFT 注入真实 60 个文本层的 410 个投影，只覆盖 attention 和 MLP，不扩到视觉/音频塔。标准 r16、122,429,440 个可训练参数、820 个 A/B 张量，训练参数保持 FP32；底座和主要计算为 BF16。当前 C03 alpha16、dropout=.05，新 A 随机/新 B 全零，新 AdamW，无 C01/C02 父件。旧 C01/C02 alpha32 是另一个合法配置，不能因后来 alpha16 就追认旧设置非法。[注入与范围检查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s032)

### 5.2 为什么从 DDP 转为选择性 FSDP2

DDP 每卡复制整个底座，增加卡数没有增加单卡容量。历史四卡 DDP 即使保留 MLP 分块，最长样本仍在 RMSNorm 等处 OOM；不能靠缩短样本或删断言完成长序列合同。因此改为只分片 60 个文本 decoder 的冻结浮点权重，`reshard_after_forward=True`，把主要底座存储分摊到四卡。

LoRA 保持原 FP32 `Parameter` 副本，不交给 FSDP 分片；初始化先核各 rank 名称/shape/dtype/顺序，再逐 tensor 从 rank0 广播。每个全局批完成后，按固定顺序、有界 bucket 对 LoRA 梯度 `SUM/world` 一次，并统一有限性检查和裁剪。不能假定 `ignored_params` 会自动同步这些参数。[FSDP2 实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s033) · [LoRA 梯度归约](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s034)

共享 embedding/head 保持复制：selected CE 在不同 rank 按正 token 数调用不同次数的 head，如果将其放入 FSDP collective，可能出现通信次序不一致。无自有参数组的 `TrainingObjective` 作为根，管理 decoder 的前反向钩子；保留共享关系并检查没有完整冻结参数旧引用残留。冻结底座仍要传递梯度到 LoRA，不能将整个 decoder 放进 `no_grad`。

QAT 解压后的模块还可能保留 `weight_shape` 这类 int64 `Parameter`。实现不把它强制转浮点或删除，而是将冻结非浮点状态排除于分片，保持对象、dtype 和数值，浮点 scale 继续分片，并保存分片前参数目录。已有小 CT 生命周期检查和实际 C01/C02 执行支持这一兼容路径；最初现场没有参数名时，不能倒称已锁定具体触发参数。[非浮点状态处理](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s035)

### 5.3 长序列：减少中间张量，不删因果上下文

MLP 按 token 维切 512-token 块，仍调用真实 Gemma/PEFT gate/up/down forward，拼回原位置；块内用 non-reentrant checkpoint，保留 RNG，外层 decoder checkpoint 也保留。它针对逐位置独立的 MLP，不将 attention 或训练序列切成互不相关的样本。固定分块的前向/重算随机状态必须一致，但分块与未分块的 dropout 随机分配不承诺逐位一致。[MLP 包装](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s036)

训练 attention 按 query 块构造 causal/局部窗口 mask，通过 PyTorch SDPA 和重算处理，避免一次构造整张 L×L attention 矩阵。限制为无 padding 的 batch1 完整文本序列、attention dropout=0，不支持任意 cache/多模态 mask；GQA 的 K/V 有显式展开，因此不能称为通用 FlashAttention 替代或无额外内存开销。[attention 范围](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s037)

selected CE 保留完整底座前向，只在有正标签的位置取前一位置 hidden state，按 head chunk=64 计算 vocabulary logits/FP32 CE，并保留模型 softcap。这样避免为所有前缀位置生成全词表 logits；它没有把上下文缩成只剩目标。设计参考了公开训练 kit，项目按当前模型、mask 与梯度合同实现并检查，不能包装为首创。[selected CE](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s038) · [来源说明](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s039)

### 5.4 全局 batch、尾批与三种 loss

四卡每卡 microbatch1、accum2，global batch=8。真实批集合不变，批内按长度分配至 rank；正常全批训练目标为：

\[
L_{batch}=\frac{1}{8}\sum_{j\in B}m_j\,CE_j.
\]

尾批 6 行按 2/2/1/1 分配，尾批 2 行按 1/1/0/0；仍用约定的分母 8，不补样、不丢样。FSDP 每 rank 需要同样次数的 decoder 通信，空缺 microstep 执行两 token 的零 loss 通信槽，并保存/恢复 RNG；不把空槽算作训练行或实际样本访问。最后小批的优化强度因此小于满批，这是显式保留的归一化约定。[训练归约与日志](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s040) · [通信槽](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s041)

旧 `action` 只监督合格动作，旧 `joint` 在存在说明的行内按行动 .75/说明 .25 分量均值；C03 `completion` 对整个合格当前 completion 的正 token 求平均，不再套 .75/.25。action/reasoning 诊断不改变梯度目标。C03 说明的实际目标系数为 27.3905%，它不是简单 token 占比，也不是实测梯度贡献。[三种 loss](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s042)

### 5.5 native 820→serving 840：架构差异的显式适配

HF Gemma full-attention 层共享 K/V，真实训练只有 410 个投影、820 个 A/B 张量。比赛 vLLM 使用独立 K/V 槽，导出逐个将 native FP32 转为 BF16，并在 10 个 full-attention 层把 K 的 A/B 各复制一份给 V，共增加 20 个张量，成为 840 张量/420 服务投影。新增项不是额外训练参数，也不是合并了两个 adapter。

导出核精确键集合、rank、shape、有限值、原配置 targets、20 个复制项和保存回读，保留 native 原件。随后还需 SDK 编译、真实 adapter 路由和 vLLM 短加载数值差异；文件能打开不能证明服务真正使用了它，短加载差异也不能证明解题能力改善。[导出实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s043) · [C02 实际接纳](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s044)

### 5.6 已经跑通的训练与 C03 待验证内容

C01 实际 498 更新、1990×2 行消费，最长 19,356 两累积通过，训练约 10.30h；C02 实际 524 更新、2090×2 行消费，最长 26,979 两累积通过，loop 13.594h。C02 在指定训练测量阶段 allocated 峰 52.7096GiB，最长检查 reserved 59.45–62.85GiB；这些不是全生命周期总显存峰值，也不能与不同工作量运行直接比较加速比。

C02 训练样本均值 loss .226930→.192685，最终张量有限且发生变化、真实服务短加载通过；部分行为评测仍出现回退。证据说明训练工程和产物有效，不说明配方有效。[C01 实际训练](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s045) · [C02 实际训练](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s046)

C03 固定一 pass，U=ceil(990/8)=124，W=ceil(.05×124)=7；更新 j≤W 应用 peak×j/W，之后按冻结 horizon 单次 cosine 到 0.1peak，即 1e-5。保存点为 ceil(U/4)、ceil(U/2)、ceil(3U/4)、U，即 31/62/93/124，均为 native FP32。参数来自公开研究与项目诊断的起始方案，尚无本项目 LR/alpha/调度独立消融证明最优。[调度定义](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s047)

当前还实现了 final AdamW、各 rank RNG、cursor 与冻结绑定留存，但 `resume_entry_implemented=false`。加载旧 adapter 再新建 AdamW 是新阶段 SFT；不能称为原训练精确断点续跑。[状态留存边界](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s048)

## 6. 实验方法与负面复盘

### 6.1 用可比实验回答有限问题

prompt、工具集合、命令时限、会话时限、总输出和思考预算都被探索过；多项同时变化时只能评价整个版本。4096 thinking 与 4096 总输出的组合曾出现纯说明到 length、没有工具动作的响应，支持检查动作空间，不能推出某个新上限必然提分。社区分数只在源码、配置与 adapter 身份可绑定时才可用于对应结论，不将作者最佳分嫁接到公开脚本。

C01 的同提示同温度对照是无 adapter T0.2=51/129，LoRA T0.2=48/129，保持46/新增2/丢5；同 LoRA T1.0=52/129，但缺当前同配置的无 adapter T1 对照。旧 9 工具/420 秒版的 T1=49 不充当这个缺失控制。调用缺必填 107→1、连续重复 81→12 有改善，同时空 patch 11→23，说明协议指标与完整修复目标可能分离。[完整开发对照](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s049)

线上 C01 T0.2=.17 有既有 API 完成观测；T1=.15 为最新用户截图观察，本次没有重新 API 核 ID。它们与本地 52/129 是不同指标，不能换算成 Kaggle 分数或声称温度稳定优势。[线上证据范围](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s050)

### 6.2 续训退化推动 fresh 方案，但没有唯一根因

C02 从 C01 native 权重精确继承、新建 AdamW，工程训练完成；主动中断的行为窗口只有 T0.2 2/10、T1 1/6，而 C01 同窗分别 4/10、3/6。该窗口是 FastAPI-only、非随机、已暴露开发题，不能外推完整129分数，不能确诊灾难遗忘、LoRA容量不足或某个 LR 必然有错。

fresh C03 同时改变了数据重选、来源、均权、completion 目标、alpha、LR、pass 数和调度，并增加 checkpoint 选择。即使未来表现改善，也只能先归属于这组联合方案；独立因素需要再做固定其余条件的对照。[部分评测限制](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s051) · [研究合并限制](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s052)

### 6.3 检查点×温度：先保留退路，再选择候选

面板计划用 base、step62、step93、step124，在同 12 题、同顺序、同 prompt/工具/预算下各跑 T.2/T1；每臂从原快照开始。25%权重保留但不在该面板中。观察总解题数之外，还看逐题新增/丢失、空 patch、额度、耗时及必要动作轨迹。

这能在一个训练 horizon 内比较中间权重和末版，但 checkpoint 与固定 GPU 有绑定，单轮也存在采样/负载差异；12 题很小且已暴露。中途 step62 的 cosine LR 历史也不等于另一个独立半程 horizon 训练。面板不证明隐藏泛化、不自动选 final、不自动触发129或提交。[面板设计](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s053)

## 7. 交付与失败处理

统一 CLI 组织 `fresh-data → audit-data → freeze → train-package`，并提供 `eval-package`、`checkpoint-panel-package`、`lean-return/restore-return` 等入口。注册、审查、freeze、目标机执行与结果接纳是不同状态，配置/包生成不能提前写成训练或评测完成。[入口](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s054)

压缩发行用 manifest 绑定精确成员、大小及原字节，服务器先 CRC 和成员集检查，拒绝路径逃逸、重复/大小写冲突、符号链接/目录联接和不同现有文件；只补缺或复用已相等文件，展开后再次逐流比对。CRC、大小用于完整性辅助，不能代替直接字节相等，也不是数字签名或供应链认证。[安全展开](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s055) · [实际部署绑定](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s056)

失败保留原始退出码、partial/not_run、源绑定和清理事实，不用外包装 exit0 代替业务结果。曾发生 vLLM IPC 路径 120B 超过 AF_UNIX 107B 限制、两臂启动均零题；修复改用官方 `VLLM_RPC_BASE_PATH` 的短独占目录，先证实零题与清理闭合再继续未执行授权，不拿它当解题失败或无限重试入口。[IPC 修复](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s057) · [既有根因记录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s058)

面板四 worker 独立；每卡 T.2 清理闭合后才进入 T1，单卡失败隔离，未闭合自有进程阻止该卡切温和封包。CPU 失败且 GPU 未动可如实 PRESERVED；不伪造恢复证据。这是共享算力上的资源生命周期控制，不是业务容错或跨机器高可用。[worker 与封包检查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s059)

回传瘦身保留全部实际 HTTP/ADK/patch/验证与源码差异，只在 producer 对完整内容直接比较相等后引用不可变输入；嵌套 ZIP 保留成员字节，不宣称恢复原压缩编码。C01 原包 1,706,923,496B → analysis 154,622,101B，减少90.94%，21,093个 HTTP 文件/7,031组和1,030个 ADK/observations 文件保留原字节。它是重复输入消除的实测传输收益，不是通用压缩比。[瘦身结果](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s060)

C03 中间 native 权重、optimizer/RNG 默认留服务器；面板 analysis 返回完整行为轨迹，但未返回的中间输入 ZIP 明确标 NOT_RETURNED。没有拿到确切输入就不能声称本地完全恢复或直比中间权重；选中后再取准确候选与温度的输入，或用 full 模式。[面板回传限制](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s061)

## 8. 可讲清的贡献与真实短板

适合对外呈现的贡献是执行/评分分离、原始轨迹与实际动作的绑定、学生状态上的受控纠偏与资格合同、原生 token 边界、长序列选择性分片和服务 K/V 适配，以及能复查失败的压缩交付。论文和社区训练 kit 提供动机或实现参考，项目的具体适配和验证范围应逐项说明。[来源与依赖归属](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s039)

当前短板也明确：没有稳定、独立的泛化提升结论；本地开发集暴露且外部留出环境未就绪；C02 仅部分行为窗口；C03 训练和检查点对照待回传；缺多 seed 和单因素消融；精确 resume 入口未完成；部分历史资源恢复只有操作回执、缺独立 after 原表。项目已执行的是 SFT 与离线受控采集，没有执行 GRPO、DPO 或在线 RL。

可对外主张逐项见[证据与指标表](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/evidence-and-metrics.zh-CN.md)，岗位表达见[简历与面试表达](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/interview-guide.zh-CN.md)，可复制的公开文章骨架见[公开展示方案](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/release-notes.zh-CN.md)。


# 证据与指标表

> 核对日期：2026-11-16。用途：在简历、作品集或面试中，每个数字和技术主张都能回到明确证据。本次仅定向阅读源码与已有报告；没有重新执行历史训练、全量 token 审计、模型加载、任务判分或权重逐值验算。表内“已验证”指相应既有报告记录的验证。

## 1. 如何读这份表

- **已实现**：当前源码存在相应路径，不自动等于目标机通过。
- **CPU/发行已验**：本地数学、小模型、数据合同或打包检查有记录，不替代 31B/CUDA 或官方任务执行。
- **历史目标机已验**：具体 C01/C02 运行已有回传和接纳，只覆盖对应配置。
- **数据已冻**：来源、选择、模板/mask 和快照通过冻结合同，不代表数据对行为必有收益。
- **用户报告**：用户转述或截图观察，没有在本次重新获得独立原件核验。
- **待验证**：链路/计划已准备，实际目标行为或完整恢复尚无证据。

每条主张有 E 编号；[小型机器索引](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/evidence/claims.json)保存主要证据的项目相对路径、定位文本和核对时行号，不包含权重或成批轨迹。源码链接指本次整理的脱敏摘录，原行号属于本地证据快照；历史行为结论同时链接到对应报告，不能把当前源码当作所有旧运行的执行身份证明。

## 2. 系统执行与可观测性

| 编号 | 可对外主张 | 状态、版本与指标 | 源码/报告锚 | 限制与推荐说法 |
|---|---|---|---|---|
| E01 | 围绕官方 SDK 建立本地编码 Agent 实验与取证链路 | 已实现、历史目标机已验；当前学习线以 r04 六工具为输入身份 | [Evaluator 接线](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s007)、[观测层边界](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s013) | 官方 ADK 循环、工具与独立判分来自上游。可说“接入与适配官方执行/评分”，不要说“自研官方裁判”。 |
| E02 | 固定任务预算、采样和提交身份，支撑可复现对照 | r04：28计费调用/480s/80轮；命令240s、验证300s；8192总输出/4096思考；p=.95/k=64 | [冻结配置](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s006)、[C03合同](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s062) | 六工具是项目选择，非官方工具上限；免费 status/submit 与计费分开。当前活动配置不替代冻结输入。 |
| E03 | 通过 HTTP→ADK→官方 trace 关联模型生成与实际执行 | 历史 C01 双臂：7031请求、6464工具动作、258题臂有审查 | [被动 HTTP relay](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s012)、[覆盖报告](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s014) | 覆盖是对应运行的动作/路由绑定，不是所有长 reasoning 逐字深读，也不是服务吞吐指标。 |
| E04 | 保存官方压缩前后证据并核实摘要送达 | 已实现；interval15/overlap2/threshold14336/retention5；C01可解析摘要167/141均送达 | [ADK配置](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s009)、[真实摘要观察](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s010) | compaction 来自 ADK。送达不等于准确，C02有摘要状态失真；不要宣称“无损压缩”。 |
| E05 | 用空补丁/参考补丁控制限定本地评测适用性 | 129任务×2控制=258题次；117参考可过=115判别+2空补丁可过；12限制 | [校准原件](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s063)、[口径解释](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s015) | 复用既有校准，本次未重跑。原始129分母保留；不声称隐藏镜像等价。 |
| E06 | 分离 Agent 会话时长、任务总时长与官方结果 | 已实现，历史逐题原件可核；独立保存patch/verification/summary | [时间与结果字段](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s064) | HTTP墙钟包含等待/prefill/生成/传输，不能直接称纯decode或私人思考时间；包装exit0不等于全题成功。 |

## 3. 轨迹、教师与训练数据

| 编号 | 可对外主张 | 状态、版本与指标 | 源码/报告锚 | 限制与推荐说法 |
|---|---|---|---|---|
| E07 | 用可重叠错误分类指导下一状态与训练目标 | 已实现并用于既有轨迹复盘；调用表示、行动时机、语义覆盖分开 | [分类口径](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s017)、[具体失败形态](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s018) | 分类数不等于互斥主因，也不是“可挽回题数”；未知语义原因保留未知。 |
| E08 | 从真实学生请求边界进行受控教师接续 | 已实现、既有真实采集；请求/状态比较、实际唯一ADK调用和回执绑定 | [执行绑定](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s019)、[实际输入与提案](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s020) | 独立离线重建，非暂停原比赛时钟；RECONSTRUCTED不说成原现场逐字节续跑；本次无新教师。 |
| E09 | 把局部有效动作与整题修复资格分开 | 合同与导出已实现；局部lane、自然闭合TaskResult、review来源分别记录 | [短片段合同](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s065)、[自然链合同](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s023) | 工具ok不等于语义正确；无独立TaskResult的fragment不能记FULL_REPAIR或学生新得分。 |
| E10 | 使用动作前公开短决策说明，拒绝后验改写进入监督 | 已实现；TEACHER_PUBLIC_DECISION_SUMMARY；私人thinking未采集 | [CLI协议](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s024)、[来源字段](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s066)、[主SPEC](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s067) | 可以讲“公开决策说明监督”；不能讲“获取了教师内部思维链”或“思维链蒸馏已证提分”。 |
| E11 | 用真实模板/tokenizer严格定位当前目标的监督边界 | 已实现；21个既有真实serving渲染fixture保持；D05全990行/12199343位置已有独审核验 | [文本/token前缀](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s068)、[D05独审范围](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s069) | prefix、历史、工具回执标签均-100；完整前缀仍参与前向。该规模来自既有独审，本次未重跑全量审计。 |
| E12 | 为C03增加完整合格completion CE，保留旧action/joint | 已实现、CPU已验；多call全资格、action/reason不重叠且覆盖当前全部正token | [Native完整目标](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s025)、[loss分支](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s042) | 未合格当前说明先显式移除再渲染，不能把它masked后留作答案提示；不是旧数据回改。 |
| E13 | 解决A转换来源工具回执后的模板边界差异 | 数据已验；18字符thought入口；14轨迹/382源目标严格重建、选中37 | [修复原因与全量机械核对](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s028)、[源码](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s070) | 382是源目标结构检查数，非新增训练行；未改变当前训练模板，不是SDK重执行。 |
| E14 | C03冻结多来源、多个行动阶段的990目标回合 | 数据已冻：本地720/113任务，Native233/41，A37/6，合计990/160；外部47任务/45repo | [最终统计](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s071)、[来源拆分与重叠说明](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s072) | 990不是独立题、不是完整轨迹；本地Gemma245/Sol475的任务覆盖存在交集。 |
| E15 | 对训练token、长度和来源暴露做明确核算 | 12199343输入位置；199458正token=156980行动/边界+42478说明；最长26910，最大completion2208；0截断 | [统计](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s073)、[真实summary](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s074) | 训练上下文上限32768与推理输出上限8192分开；summary的初始状态字段不单独决定冻结结果。 |
| E16 | 按题/阶段均衡优化目标，避免长链天然占更多权重 | 数据与CPU已验：m=N/(T*K_i*n_i_s)，平均1；阶段313/271/195/211；倍率.1546875–3.09375 | [权重计算](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s030)、[统计及系数](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s075) | 均匀无放回遍历，不叠加加权重采样；说明27.3905%是目标系数，非梯度贡献；尚无行为最优证明。 |
| E17 | 外部数据按仓库留出，明确原生与转换来源边界 | 已审选/未实跑留出：train47任务/85attempt/1532回合；dev/final各8repo9issue未训练 | [审选范围](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s076)、[留出限制](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s077) | Native=SOURCE_NATIVE_CAPTURE_SUPPORTED，A=SOURCE_CONVERTED_SUPPORTED；均不冒称本SDK新执行。留出环境尚未重建，本地129已暴露。 |

## 4. 训练、长序列与导出

| 编号 | 可对外主张 | 状态、版本与指标 | 源码/报告锚 | 限制与推荐说法 |
|---|---|---|---|---|
| E18 | 恢复官方QAT权重为冻结BF16底座，只更新PEFT LoRA | 已实现、历史GPU已验；run_compressed=False，冻结恢复后底座，关闭QDQ wrapper | [加载/冻结实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s031)、[固定版本来源](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s078) | 不是重新预训练、全参微调或新NF4量化；恢复的是原QAT包权重。 |
| E19 | 精确限定文本LoRA范围，防止宽后缀扩到其他塔 | 已实现、历史GPU已验；60层410投影/820A-B张量/122429440参数；FP32 adapter | [模块枚举与PEFT](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s032)、[序列化scope](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s079) | C03 r16/a16/.05；C01/C02 r16/a32/.05为不同配方。没有执行rsLoRA或更大rank消融。 |
| E20 | 对冻结decoder做FSDP2分片，LoRA复制并显式归约 | 已实现、历史四A800已验；60decoder分片，rank0初始广播，LoRA SUM/world，每global批一次 | [分片与身份检查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s033)、[有界bucket归约](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s034) | FSDP2为PyTorch能力；项目贡献是选择性分组、同步及执行边界，不说发明分布式训练框架。 |
| E21 | 处理混合dtype与QAT残余非浮点Parameter | 已实现、生命周期检查及历史训练已验；非浮点冻结状态复制保留，浮点scale仍分片 | [非浮点兼容](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s035)、[历史问题与边界](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s080) | 不删/不转dtype求通过；最初缺现场参数名，weight_shape只能称强候选，不能倒写确定根因。 |
| E22 | 让共享embedding/head留在FSDP外，避免动态head调用次数引起collective错序 | 已实现、历史GPU已验；无自有参数组objective root，保共享关系 | [设计说明](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s081)、[root/head检查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s082) | 是针对本selected CE路径的工程设计，非通用最优分片方案；冻结decoder仍需传递输入梯度。 |
| E23 | 通过MLP分块重算和query分块attention保留完整长输入 | 已实现、CPU数学/历史GPU已验；MLP512，non-reentrant/preserve_rng；query512，完整因果/局部窗口 | [MLP实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s036)、[attention实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s037)、[参考来源与限制](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s083) | 不截断因果序列；不承诺分块与未分块dropout逐位同；attention只支持既定无padding/batch1文本场景，有显式GQA展开。 |
| E24 | 仅在正标签位置计算分块vocabulary CE，降低logits开销 | 已实现、CPU值/梯度及历史GPU已验；head chunk64、因果位置-1、softcap保留 | [selected CE](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s038)、[来源归属](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s084) | 完整底座前向仍在；设计参考公开kit，不称原创算法或无工作量控制的加速收益。 |
| E25 | 全局batch与任意尾批保持真实样本消费和通信顺序 | 已实现、CPU及历史GPU已验；4×micro1×accum2=8；尾6为2/2/1/1，尾2为1/1/0/0；保留分母8 | [微步/归约](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s040)、[零loss通信槽](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s041)、[C02实际消费](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s044) | 不补样、不drop-last；通信槽不计训练行且恢复RNG。尾批更新比满批小，是显式约定。 |
| E26 | 按Gemma共享K/V架构显式转换训练与服务adapter布局 | 已实现、C01/C02真实导出与短加载已验；native820FP32→serving840BF16，增加20个K→V复制项 | [导出函数](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s043)、[C02导出/加载](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s085) | 840不代表额外训练；需保存回读、SDK/路由和真实vLLM检查；短token数值变化不等于任务提分。 |
| E27 | 用目标最长样本、真实首更新和全行日志验证训练不是空跑 | 历史C01：1990×2/498更新、最长19356两累积、loop10.30h | [C01接纳表](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s045) | 首次load失败与后续成功分别保留；不能称全过程首次成功。C01测量窗口allocated43.84GiB不是推理峰值。 |
| E28 | C02续训确实继承父权重、更新并完成固定样本消费 | 历史C02：2090×2=4180消费/524更新；最长26979两累积；loop13.594h；真实短加载 | [C02执行/参数表](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s046)、[参数变化](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s086) | 新AdamW阶段，非optimizer精确续跑；820张量均变≠每个元素均变。ΔL2/L2=.1259964不是能力提升12.6%。 |
| E29 | 明确记录loss、梯度与显存的测量口径 | C02样本均值.226930→.192685；末步全局.00413465；515/524裁剪前norm>1；指定窗口allocated52.7096GiB、longest reserved59.45–62.85GiB | [loss解释](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s087)、[梯度解释](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s088) | 不是验证loss、不是全局归零；裁剪前峰值7984.323不是裁剪后更新幅度；无同工作量性能对照，不报加速倍数。 |
| E30 | C03实现按真实optimizer步数冻结的一pass调度与四快照 | CPU/发行已验、目标训练未接纳：990/global8→124更新；warmup7；peak1e-4→1e-5；31/62/93/124保存 | [调度合同](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s047)、[冻结规格](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s089) | 不能拿旧两pass的中点当独立一pass cosine；不是经本项目消融证明的最优LR/调度。 |
| E31 | 留存final optimizer/RNG/cursor，为后续恢复提供状态基础 | 已实现、CPU已验；final一份AdamW，各rank RNG，输入/调度绑定 | [留存实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s048)、[实现报告](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s090) | `resume_entry_implemented=false`；不能写“支持精确断点续训/故障自动恢复”。中间只有adapter快照。 |

## 5. 实验结果与未完成事项

| 编号 | 可对外主张 | 状态、版本与指标 | 源码/报告锚 | 限制与推荐说法 |
|---|---|---|---|---|
| E32 | C01改善部分协议指标，完整任务收益尚未建立 | 完整129开发：baseT.2=51，C01T.2=48，C01T1=52；同温度保持46/新增2/丢5；缺必填107→1、连续重复81→12、空patch11→23 | [结果与分母](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s049) | 全部为暴露开发题；缺当前baseT1控制；不写“LoRA提升解题率/稳定超过底座”。协议改善不代替任务成功。 |
| E33 | C02完整训练有效，但有限行为测试出现回退信号 | 主动中断：T.2=2/10，对C01同窗4/10；T1=1/6，对3/6；另2活动题中断，118/122未跑 | [部分结果](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s051) | 非随机FastAPI-only窗口，不补成129成绩，不确诊灾难遗忘、rank不足或GPU问题。 |
| E34 | 正确区分本地开发计数与Kaggle线上分数 | C01T.2=.17有既有API观测；T1=.15为2026-11-16用户截图，截图未显示ID，本次未API重核 | [T.2完成观测](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s091)、[最新截图记录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s050) | 52/129不是.52或线上分数；不写奖牌、.22、排名或相对线上提升；无C03线上结果。 |
| E35 | C03 fresh重启由数据/方法复盘推动，明确联合变化与状态 | 数据已冻/发行完成；仅用户报训练23/124、loss1.25→.35、LR9.6e-5 | [动态状态](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s001)、[状态JSON](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s002)、[研究边界](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s052) | 不写已完成124步。数据/目标/alpha/LR/pass/调度同时变，未来改善不能单因素归因；未执行GRPO/DPO/在线RL。 |
| E36 | 预先定义checkpoint×温度面板并实现真实base与adapter分支 | CPU/发行已验、目标NOT_RUN：4候选×2温度×12=96题次；base/step62/93/124；25%只保留 | [面板设计](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s003)、[真实导出与检查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s092) | 12题已暴露；checkpoint与GPU绑定、单次采样，不是独立泛化/多seed；final未被选为最佳；面板不自动执行129/提交。 |

## 6. 交付、恢复与来源归属

| 编号 | 可对外主张 | 状态、版本与指标 | 源码/报告锚 | 限制与推荐说法 |
|---|---|---|---|---|
| E37 | 用统一CLI和manifest管理数据、训练及评测交接 | 已实现；C03训练7件67227657B/63ZIP成员；面板5件、source ZIP12509850B/65成员 | [CLI](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s054)、[训练发行](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s093)、[面板发行](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s094) | 交接完成≠目标执行；入口不是自动无限采集/训练/发布闭环。成员数量必须说明所属ZIP。 |
| E38 | 安全展开后逐流直比，拒绝不同文件覆盖和路径逃逸 | 已实现、既有CPU/发行已验；成员集/类型/大小/CRC/链接/目标包含关系与最终字节检查 | [stage](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s055)、[部署合同](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s056) | 不是密码学签名或远程可信证明；CRC/大小不能单独证明内容身份；授权环境小修须保before/after/diff。 |
| E39 | 隔离失败、保护同卡切温顺序并保留真实partial状态 | 已实现/CPU已验；原IPC120B>107B零题失败已定位；短RPC目录；面板单卡失败隔离 | [IPC分配](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s057)、[worker清理检查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s059)、[面板失败边界](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s095) | 只清可核归属的自有进程，不重置他人GPU；历史恢复缺after原表的地方不能补成“全证据齐”。 |
| E40 | 通过精确重复引用减少回传，同时保留行为证据 | 历史全内容验证：1706923496B→154622101B，减少90.94%；21093HTTP文件/7031组、1030ADK/observations原字节保留 | [瘦身实测](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s060)、[实现](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s096) | 仅对应包实测；依赖冻结输入才能恢复引用；ZIP成员字节≠原容器编码；未知/不同内容不省。 |
| E41 | 明确服务器留存权重与本地可恢复范围 | 面板实现并已交付；analysis全轨迹回传，中间输入/权重SERVER_RETAINED/NOT_RETURNED，full另选 | [回传限制](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s097)、[pack声明](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s098) | 不能用final替代step62/93，也不能把producer收据称本地已直比服务器全树。 |
| E42 | 保留官方能力、社区参考、数据及仓库许可证归属 | 来源记录已存在；公开母集元数据CC BY4.0；仓库源码/模型/依赖各自声明适用 | [实现来源](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s039)、[包NOTICE](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s099)、[母集许可记录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md#s100) | 来源许可不自动覆盖模型、全部仓库或教师材料；本次未作发布授权认定，公开规则另标未核实。 |

## 7. 几个最容易写错的数字

| 容易混淆的表述 | 正确解释 |
|---|---|
| 990条数据 | 990个合格目标回合、160个任务ID；同题可以有多阶段、多前缀行。 |
| 199458个token | 当前正监督token；完整输入位置是12199343，不能二者互换。 |
| 820→840 | 训练与服务的张量布局转换；20个K/V复制项不是额外训练能力。 |
| 410投影 vs 420投影 | 训练真实共享K/V结构为410；服务独立槽展开为420。 |
| 27.3905%说明 | 依行内CE与任务/阶段倍率算得的目标系数；不是梯度、训练时长或数据行占比。 |
| .226930→.192685 | C02同一训练过程两epoch真实样本均值；不是固定checkpoint验证loss。 |
| 52/129 | C01 T1本地暴露开发结果；不是Kaggle public score，也不是同温度base增益。 |
| 23/124 | 用户报告的C03中途进度；不能写124更新已完成。 |
| 96题次 | 已交付检查点面板的计划执行量；不能计入已完成实验总数。 |
| 90.94% | 某份回传因重复权重/输入消除得到的压缩传输收益；不是训练显存下降或推理加速。 |

## 8. 证据冲突的处理方式

新动态规格和追加状态解释当前进度，旧报告保留原时点。D05的导出summary仍可能写 `EXPORTED_PENDING_NATIVE_MASK_AUDIT`，要结合独立审查、acceptance/frozen快照及后续状态，不能把早期字段单独当当前终态。同理，C01训练接纳报告中的线上PENDING已被后续完成观测补充，但原件不回写。

本次只读比较发现，当前 `agents/self_harness_v17` 的agent.yaml和prompt与r04冻结ZIP相同，eval_config.yaml和sampling.yaml不相同；因此历史预算/温度引用冻结ZIP、轮次配置与实际运行报告。这个比较没有改动任何文件，也没有据此认定活动配置错误。

证据证明的是项目能力和某次实验，不自动证明个人独立完成范围。简历动词须按本人真实分工选择；缺少职责、团队规模、任职时间、用户规模或上线范围时，一律不补造。


# 简历与面试表达

> 这是有证据边界的表达素材，不是已经确认个人职责的履历。默认写“参与构建/参与实现”；本人确实负责某模块后，可改为“负责”，确实决定架构与推进交付后才用“主导”。现有项目材料不能证明独立完成、团队人数、任职时间、线上用户量或业务营收。
>
> 所有数字截至2026-11-16。C01/C02已有实际训练；C03只有用户报告训练23/124步，96题次面板尚未确认执行。详细锚点见[证据与指标表](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/evidence-and-metrics.zh-CN.md)。

## 1. 项目名称与定位

推荐名称：**Gemma编码Agent评测与轨迹监督学习工程**。

项目说明：面向公开软件仓库issue修复，接入官方SDK和独立判分，构建原始轨迹取证、学生前缀纠偏、合格目标SFT、长序列LoRA训练和候选行为对照链路。

技术栈按投递岗位选取，不必全部堆入一行：Python、PyTorch、Transformers、PEFT/LoRA、FSDP2、vLLM、ADK、Safetensors、Jinja2/tokenizer、Linux进程与压缩发行。A800是实际实验硬件，不代表线上集群规模。

## 2. Agent工程岗：四条简历bullet

- 参与构建基于官方SDK/ADK与vLLM的编码Agent实验链路，固定六工具、预算及采样配置，分离源码修复与独立判分，支持版本化对照和失败回放。[E01–E06]
- 参与实现HTTP、ADK工具调用与回执的关联审计，在一次完整双温度开发实验中核对7,031个请求与6,464个动作，定位格式、行动时机、验证覆盖和摘要失真问题。[E03、E04、E07]
- 参与搭建真实学生请求边界上的受控教师接续和逐动作资格审选，将多来源数据整理为990个目标回合/160个任务ID，保留来源、原生mask及任务/阶段权重。[E08–E17]
- 参与实现manifest驱动的压缩交付、失败隔离和精确重复引用回传；在一份实际回传上将体积由1.707GB减至154.6MB，全部HTTP/ADK行为证据保留原字节，减少90.94%。[E37–E41]

如果只放三条，可合并第一、二条，把990个目标回合留给有训练职责的岗位。不要将“核对请求数”包装成日调用量，也不要把开发实验写成生产上线。

## 3. LLM训练岗：四条简历bullet

- 参与实现Gemma31B QAT权重恢复后的冻结BF16底座LoRA训练，精确覆盖60个文本层、410个投影和1.224亿可训练参数，保留FP32 adapter与原生目标边界。[E18、E19]
- 参与适配四卡FSDP2：分片冻结decoder、复制LoRA并显式同步梯度，处理共享embedding/head、非浮点QAT辅助状态与不等长尾批；既有训练承载26,979-token完整输入的两次累积。[E20–E25、E28]
- 参与实现完整前缀上的selected CE、512-token MLP分块重算及题/阶段均衡loss；C03冻结199,458个正token，历史/工具回执不计loss，保留数据来源和留出边界。[E11–E17、E23、E24]
- 参与打通native820张量到serving840张量的共享K/V导出、保存回读与真实vLLM短加载；另实现按optimizer-step调度和四checkpoint留存，C03行为收益仍待评测。[E26、E30、E31、E35、E36]

“26,979-token两累积”来自C02实际训练；“C03四checkpoint”目前是实现和交接能力，不能改成“四checkpoint均已完成评测”。如本人未参与训练实现，只保留其负责的实验设计或审计部分。

## 4. 一分钟介绍

### 面向Agent工程岗位

我参与了一个Gemma编码Agent项目，任务是在真实软件仓库中修复issue。我重点关注从模型输出到实际源码修复的证据链：接入官方SDK和独立判分，记录HTTP、ADK调用及工具回执，再区分格式问题、行动迟滞和语义覆盖不足。训练数据采用真实学生前缀上的受控教师接续，动作需要实际执行和资格审核。项目还打通了长序列LoRA训练与服务导出。不过，我不会把工程跑通说成效果提升：同温度下C01是48/129，底座是51/129。当前新方案增加了中间checkpoint和固定开发面板，用行为结果决定是否采用。

### 面向LLM训练岗位

这个项目把Gemma31B的编码轨迹转成可审计的SFT目标。我参与的技术链包括原生模板与token级mask、按题和阶段均权，以及四卡长序列LoRA训练。训练只更新约1.224亿参数；冻结decoder用FSDP2分片，LoRA保持FP32副本并显式归约，另外处理共享head、非浮点量化辅助参数和尾批通信。历史训练已承载26,979-token完整输入，并完成820到840张量的服务导出和真实短加载。效果上我们看到了loss下降但局部行为回退，因此新方案改为fresh训练并保留检查点。C03还在等待完整回传，我会明确区分实现、训练成功和任务收益。

这两段是可替换的口述稿，不必一起讲。使用前把“我参与的部分”收窄到本人真实负责范围。

## 5. 最值得准备的技术追问

### Q1：这个项目的贡献是什么？SDK和框架不都是现成的吗？

官方SDK负责工具、主循环和独立判分；PEFT、FSDP2、vLLM提供基础能力。项目工作是把它们接成有明确数据身份和失败边界的实验系统：HTTP与实际动作绑定、真实学生状态上的受控接续、严格原生mask、选择性分片和共享K/V服务适配。selected CE和query分块也参考了公开训练kit，应注明来源。可以用具体适配和验收回答，不能声称发明上游算法。[E01、E20–E26、E42]

### Q2：怎样证明模型真的调用了工具，而不是在文本里写了工具名？

同时看HTTP中的tool_calls、ADK实际function call及function response，并比对call ID、名称、参数、回执序号。教师提案还必须在下一次官方重放里被消费，生成EXECUTED绑定。纯文本伪工具、只有计划或被拒绝的调用不能当作已执行正目标；JSON合法也不保证命令语义安全。[E03、E08、E09]

### Q3：为什么还要独立判分？模型自己跑pytest不够吗？

模型的自测可能漏掉问题入口、只覆盖一条分支，甚至依赖错误的测试修改。最终patch由官方独立验证工作区判分，Agent的自测回执只证明那个检查的实际观察。空补丁/参考补丁控制还用于识别本地环境的判别能力与限制，不能只看最终exit0。[E01、E05、E07]

### Q4：上下文压缩是你实现的吗？怎么判断它有没有造成退化？

摘要机制来自ADK，项目负责配置和可观测验证。既要看摘要是否进入后续主请求，也要核其是否准确保留当前编辑和验证状态。C01已返回的可解析摘要都送达，C02部分摘要仍有状态失真；因此不能笼统归因于漏发，也不能说压缩无损。没有单因素消融时不宣布它提升或损害了多少题。[E04]

### Q5：教师接续为什么不直接生成整条“完美答案”？

目标是覆盖学生真实到达的状态，教它从错误或不完整证据继续采取有效动作。教师只见当前官方输入，不看分析报告、未来判分或参考修复；新动作还要在真实工具环境执行。短接续可以保留有价值的局部步骤，但不能冒充整题解决。初始示范和中途接续有不同来源标签。[E08–E10]

### Q6：你们是不是采集了大模型的私人思维链？

没有。教师额外撰写动作前的公开短决策说明，描述已见证据和下一动作理由，字段叫decision_summary，来源明确标记。它不是内部思维链。源数据和学生响应中已有的可见说明也需要单独审核，未审或错误说明不进正标签；后验重写只作为诊断metadata。[E10–E12]

### Q7：怎么防止把工具回执和历史答案也训练进去？

每行由完整请求前缀和一个当前合格助手回合构成。先用真实模板、tokenizer核严格文本/token前缀，再用offset和跨度确定正标签，历史及工具回执均为-100。C03的完整completion还要求多调用全部有资格，action与说明mask的并集精确等于当前目标。不能只凭字符串长度或“从assistant位置开始”猜边界。[E11–E13]

### Q8：990行很少，为什么还做题/阶段均权？

一行是目标回合，长轨迹可能产生很多行，直接均匀行loss会放大这些任务。我们用m=N/(T*K_i*n_i_s)作为行loss倍率，让每题总质量相同、题内实际保留的阶段质量相同，均值为1；没有再做相同权重的重采样。这个设计透明且可审计，但没有证据证明它最优。数据量和来源多样性仍是限制。[E14–E17]

### Q9：QAT底座上训练，和QLoRA有什么区别？

现实现读取官方compressed-tensors QAT包，按固定版本API恢复已存的权重值，以冻结BF16参与训练，再注入FP32 LoRA。没有重新量化成NF4，也没有继续做量化感知训练；底座参数不更新。因而不能把这条路径简单称为常见4bit NF4 QLoRA，也不能称31B全参训练。[E18、E19]

### Q10：四张80GB卡，为什么普通DDP还会OOM？

DDP复制整个模型，卡数增加并不减小单卡的底座存储。长序列激活、MLP中间张量和临时计算还要叠加；项目真实失败不支持只靠多卡就推断容量足够。后来在保留完整序列的前提下分片冻结decoder，再配合MLP重算和selected CE。最终有C01/C02目标机训练支持，但不是同工作量的DDP/FSDP速度消融。[E20、E23、E24、E27、E28]

### Q11：为什么LoRA不也交给FSDP？共享head为什么例外？

LoRA规模相对底座小，保留普通FP32 Parameter有利于显式梯度同步、精确父件加载和单独导出；代价是每卡保留adapter及optimizer副本。共享head的selected CE调用次数依赖各rank正token数，放进FSDP可能造成collective次序不一致，所以它和embedding保持复制。方案针对本训练路径，不是所有模型都应照搬。[E20–E22]

### Q12：冻结参数为什么还需要反向？

冻结只表示这些权重不求参数梯度和不更新。后层loss仍需要通过冻结层的计算图向前面的LoRA传递梯度，因此不能把所有冻结decoder包进no_grad。FSDP冻结组仍涉及反向unshard/reshard，root和child状态要正常进入forward钩子。[E20、E22]

### Q13：尾批不足8行时怎么保证四卡不死锁、不偷偷补样？

真实行按既定全局批集合分配，尾6为2/2/1/1，尾2可为1/1/0/0。每个rank都完成固定次数的decoder通信，缺行的位置跑零loss通信槽，且恢复RNG，不计入行消费。真实loss先除每rank累积数，再SUM/world，保持sum(weighted loss)/8；最后小批因此不是除实际行数。日志记录真实行与通信槽分开，能核对无重复无遗漏。[E25]

### Q14：为什么是820张量训练，840张量服务？

训练端10个full-attention层共享K/V，没有独立的V投影LoRA；比赛服务端却需要独立K/V槽。导出将每层K的A/B复制给V，共增加20个张量，同时FP32转BF16，检查840键集合、原配置及保存回读。native原件保留；服务端还要实际注册、路由和短加载，不能只改配置就说已经生效。[E26]

### Q15：loss降了，为什么结果反而可能更差？

loss反映当前监督目标拟合，不直接评价完整修复、兼容性或预算内行动。C01缺必填和重复调用改善，但同温度解题48低于底座51，空patch也变多；C02训练loss下降，有限行为窗口仍回退。这可能涉及数据选择、目标、训练强度等，但现有实验不足以唯一归因。应保留底座并看逐题增丢、有效编辑和真实验证，而不是只选最低loss。[E29、E32、E33]

### Q16：fresh C03依据是什么？它证明了哪个超参更好？

它是复盘后的联合起始方案：重选多阶段目标、补公开Native/A、完整completion CE、r16/a16、一pass和warmup+cosine，并保存中间权重。相关论文和社区源码提供设计参考，不能证明这组Gemma超参最优；多个因素一起变也不能单独归因alpha或调度。现在只有23/124步的用户进度，尚无C03完成和行为收益结论。[E30、E35]

### Q17：你支持精确断点续训吗？

目前不支持完整可执行resume入口。C03实现了final AdamW、rank RNG、cursor、数据与调度绑定留存，中间点保存native adapter；但尚未实现和验证完整状态加载后继续运行。C02是加载父adapter、新AdamW的新阶段SFT，不能称原run精确恢复。[E28、E31]

### Q18：本地52/129，对应什么线上成绩？

不能换算。本地129是已暴露开发题；C01 T1为52/129，而同温度无adapter控制缺失。现有线上观测是T.2=.17、T1截图=.15，来源等级也要区分。当前没有奖牌、.22或C03线上成绩，不能把社区作者分数当成自己的结果。[E32、E34]

### Q19：90.94%回传减少是怎么做到的？会丢证据吗？

主要去掉与不可变输入完整字节相等的重复权重和ZIP成员，保留引用；不同内容和未知内容仍保存。行为轨迹、HTTP/ADK、patch和验证原字节保留。恢复依赖精确冻结输入，ZIP可重建成员字节但不承诺原压缩容器编码；C03未回传的中间输入要明确缺失，不能拿final代替。[E40、E41]

### Q20：如果再给一轮实验预算，你会优先做什么？

先接纳当前训练回传和按既定96题次面板比较base及三个checkpoint，确认没有路由或输入漂移，再用逐题行为选择候选。随后若要因果结论，在固定数据、初始化、预算和horizon的条件下选一个因素做对照，并报告多seed成本与不确定性。独立泛化需要先重建未入训练的外部留出环境；不能继续把开发题上的选择结果当盲测。这里是下一步建议，没有自动执行授权或已完成结果。[E17、E35、E36]

## 6. 一个可展开的工程案例

**问题**：完整长序列LoRA在DDP下仍然OOM，缩短样本会破坏已冻结的数据合同。

**设计**：保留原因果输入和全局batch，把冻结decoder交给FSDP2分片；LoRA复制并显式归约，head/embedding保持共享副本；用非浮点状态排除规则兼容QAT残余Parameter，尾批用零loss通信槽保持collective顺序。

**验收**：先做CPU数学与真实微型模块检查，再看目标机最长行两累积、正式首步、实际全行消费、参数变化、导出及短加载。C02回传记录524更新、4180行访问、最长26979通过，loop13.594h。

**限制**：这证明既定配置能完成训练，不证明FSDP相对DDP的加速倍数或LoRA的任务收益。部分行为窗口回退仍应保留，并推动后续检查点选择。

口述时可根据本人职责展开“我核了哪一个接口/写了哪一部分/如何发现反例”。没有个人分工记录时，不补造“我一个人解决了整个系统”。

## 7. 使用前真正需要补的信息

| 待补项 | 原因 | 目前可写方式 |
|---|---|---|
| 本人负责模块、设计决策与代码评审范围 | 项目证据不能自动等于个人贡献 | “参与构建”；按真实分工收窄 |
| 项目起止时间、团队人数、协作方式 | 本文未核实 | 暂留空，不猜人数或工期 |
| C03最终训练、加载和四快照回传 | 当前只有进度转述 | “已实现并交付，等待目标回传” |
| 96题次面板及后续候选结果 | 目前是交付计划 | 不填写通过率或最佳checkpoint |
| 外部留出环境与独立结果 | 泛化论断所需 | “按仓库留出，尚未实跑” |
| 多seed、单因素消融与同工作量性能基准 | 因果、稳定性和加速结论所需 | 不写稳定提升、最优或加速倍数 |
| 公开仓库URL与个人署名范围 | 公共文档与教学入口见本项目README | 个人署名范围仍须按真实职责确认 |


# 公开版本说明

本版本于2026-11-16整理，发布范围为技术文档、脱敏证据摘录、机器可读指标和CPU教学Notebook。GitHub与Kaggle共享同一个Notebook；新增代码仅作合成例子的教学演算。

## 阅读入口

- [技术总览](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/technical-overview.zh-CN.md)：架构、数据资格、训练与交付设计。
- [证据与指标](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/evidence-and-metrics.zh-CN.md)：42项主张、对应状态和限制。
- [技术追问与表达示例](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/interview-guide.zh-CN.md)：解释设计取舍，不证明个人独立完成范围。
- [脱敏源摘录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/source-excerpts.md)：项目相对文件名、原行号和有限摘录。
- [实验指标](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/evidence/metrics.json)：逐版本分母、来源和验证边界。
- [教学Notebook](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/notebooks/gemma-agent-trajectory-sft.ipynb)：同时镜像到Kaggle。

## 可复现范围

Notebook的四个代码单元使用Python标准库演示题/阶段均权、当前目标mask、selected CE与共享K/V键布局；输入均为合成数据，不读取任务、模型或凭据。已在本地逐单元检查。平台发布使用Quick Save，公开版本没有启动Kaggle计算任务。

完整31B训练、官方任务执行和历史HTTP审计需要对应资源与环境，不能用教学单元代替。历史训练和任务指标来自既有回传报告。本仓库保留负面结果，不声称LoRA稳定提分、隐藏泛化改善、奖牌或达到0.22。

## 实验状态

C03已冻结990个目标回合、160个任务ID。当前最后接纳的进度信息仅为用户报告23/124步；完整训练回传尚待核实。base/step62/step93/step124各T0.2/T1.0的12题开发面板，共96题次，目前仍按待执行/待回传记录。外部留出环境与精确resume入口也未完成。

这些状态限定于本次证据整理截点；后续进展须附新回传后更新，不能从旧计划推断完成。

## 来源与发布边界

官方SDK/ADK工具、独立判分、模型模板、LoRA、FSDP2和vLLM属于上游能力。query分块与selected CE等内存策略参考公开训练kit；项目工作是适配、组合、约束和验证。来源见[NOTICE](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/NOTICE.md)。

官方规则的公开代码条款可从[本竞赛规则页](https://www.kaggle.com/competitions/gemma-4-developer-agent/rules)检索到，要求在本竞赛关联的论坛或Notebook分享。因此同一份教学代码与技术材料也通过关联本竞赛的Kaggle Notebook发布。该条款不替代具体第三方材料的授权。

只发布明确列出的文档与教学材料。真实部署身份、地址、账号凭据、原始HTTP、任务数据、权重和optimizer均不属于本次附件。源摘录经脱敏，不主张与原文件全字节等价；它用于解释实现依据，不是完整可运行源码发行。


# 脱敏源证据摘录

本页来自2026-11-16的定向源码与既有报告阅读。路径为项目相对路径；行号指原始本地文件。
摘录经过路径、运维身份和无关账号脱敏，长行可能省略后半部分；它不是原文件的字节等价副本，也不能替代完整执行证据。
历史“已验证”归属于相应原报告，本次没有重新运行模型、训练、任务评测或全量token审计。

## s001

来源：`learning_pipeline/specs/FRESH_SFT_C03.md:77`

````text
77: ## D096：独立检查点面板（2026-11-16）
78: 
79: 用户报当前训练step23/124、loss1.25→0.35、LR9.6e-5，四卡健康；此为用户进度转述，尚未收到本轮训练原件，不冒称本地已接纳完成。原990行/124步/保存31-62-93-124配方和训练发行零修改。
80: 
81: 用户新增明确授权：训练正常收尾并完成原封包后，以原12题开发面板对底座、50%、75%、100%各跑T0.2/T1.0。GPU0/1/2/3依次对应四候选，四worker并行，卡内先T0.2后T1.0，每臂独立原快照，共8臂96题次。覆盖此前只首选base/50/100与T0.2的建议；12题身份和顺序未变，旧预声明保存在development_panel_initial_20261116.json。25%仅保留，未扩大为八轮129。
82: 
83: 统一入口新增`python -m learning_pipeline checkpoint-panel-package`，轻量发行`dist/harness_k_a800/checkpoint_panel_c03_r01`，服务器仅新目录展开。复用成熟官方SDK/裁判runner及原258控制证据；全129校准和12题执行范围分别核，基座4成员/无adapter与三checkpoint6成员/真实注册路由均覆盖。原生820 FP32用原导出函数转换840 BF16、r16/a16不变；所有Agent文本与预算只允许温度及adapter声明差异。短IPC、UUID→index、同卡清理后续温、卡间失败隔离保持。
84: 
85: 默认analysis回传全部轨迹和原始逐题判分，50/75输入权重保留服务器，manifest明确本地尚无外部引用，不能用final补中间点；--full可自含。根据8臂逐题矩阵、空patch/额度/耗时和必要轨迹选候选，不根据最低训练loss或一题微差自动判胜。全部为暴露开发题，无隐藏泛化/线上分数保证。当前交接已准备，实际面板/新129/Kaggle均NOT_RUN；训练原包与所有历史不变。
````

## s002

来源：`reports/fresh_lora_implementation_20261116/status.json:1`

````text
1: {
2:   "schema": "fresh-sft-execution-progress-v1",
3:   "date": "2026-11-16",
4:   "decision": "TRAINING_USER_REPORTED_IN_PROGRESS_PANEL_HANDOFF_READY",
5:   "spec": "learning_pipeline/specs/FRESH_SFT_C03.md",
6:   "research": "COMPLETE_6_INDEPENDENT_REPORTS",
7:   "local_selection": {
8:     "status": "COMPLETE",
9:     "reviewed_tasks": 129,
````

## s003

来源：`reports/checkpoint_panel_c03_20261116/overview.md:3`

````text
3: 2026-11-16。用户最新授权为**四候选×两温度×12题，共96题次**，覆盖先前三候选/单温度建议。原C03训练和124步/四快照配方不动；用户报step23/124、loss1.25→0.35、LR9.6e-5，尚未收到训练原件，本地没有宣布训练完成或提分。
4: 
5: |GPU|候选|顺序|
6: |---|---|---|
7: |0|官方QAT底座，无LoRA|T0.2 → 清理 → T1.0|
8: |1|step62 / 50%|T0.2 → 清理 → T1.0|
9: |2|step93 / 75%|T0.2 → 清理 → T1.0|
10: |3|step124 / 100%|T0.2 → 清理 → T1.0|
11: 
````

## s004

来源：`learning_pipeline/server/requirements_train.txt:1`

````text
1: torch==2.10.0+cu128
2: transformers==5.13.1
3: peft==0.21.2
4: accelerate==1.15.0
5: compressed-tensors==0.15.0.1
6: tokenizers==0.22.2
7: safetensors==0.8.0
8: Jinja2==3.1.6
````

## s005

来源：`baseline_settings.py:109`

````text
109: EXPECTED_VERSIONS = {
110:     "swegemma": "0.2.11",
111:     "adk-submission": "0.2.13",
112:     "adk-eval-core": "0.1.0",
113:     "google-adk": "1.36.1",
114:     "google-genai": "2.11.0",
115:     "fastapi": "0.136.3",
116:     "starlette": "0.52.1",
117:     # Existing host dependencies used by both the service and visible tests.
````

## s006

来源：`learning_pipeline/configs/r04_cycle01.json:14`

````text
14:   "expected_evaluation": {
15:     "verification_timeout_seconds": 300,
16:     "timeout_seconds": 240,
17:     "max_time_minutes": 8,
18:     "max_tool_calls": 28,
19:     "max_turns": 80
20:   },
21:   "expected_sampling": {
22:     "temperature": 0.2,
````

## s007

来源：`runtime/baseline.py:302`

````text
302:         evaluator = Evaluator(EvalConfig(
303:             tasks_path=settings.DATA_DIR / "tasks.jsonl", snapshots_dir=settings.DATA_DIR / "snapshots",
304:             results_dir=settings.OUTPUT_DIR / "results", submission_dir=agent_dir, models=models,
305:             sandbox="subprocess",
306:             graph_dir=str(settings.DATA_DIR / "graphs"), embeddings_dir=str(settings.DATA_DIR / "embeddings"),
307:             wheels_dir=settings.DATA_DIR / "wheels", limits=limits,
308:             generation_constraints=generation_constraints, adapter_manifest=adapters,
309:             **evaluation, display_mode="quiet", enable_sandbox_testing=True,
310:             context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
````

## s008

来源：`runtime/baseline.py:343`

````text
343:                 logger.info("[%d/%d] Evaluating %s", index, len(tasks), task.instance_id)
344:                 task_started_at = time.monotonic()
345:                 result = await watch_evaluation(
346:                     evaluator.evaluate_task(task=task, task_index=index, total_tasks=len(tasks), context=context),
347:                     server, recorder, settings.OUTPUT_DIR / "vllm.log", logger,
348:                     check_evidence=check_evidence,
349:                     preserve_interrupted=lambda: evaluator.sandbox.preserve_current("interrupted"),
350:                 )
351:                 summary = {
````

## s009

来源：`runtime/baseline.py:310`

````text
310:             context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
311:             events_compaction_config=EventsCompactionConfig(
312:                 compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5,
313:             ),
314:         ))
315:         # The official manager is still empty here. The subclass only saves
316:         # Agent evidence before deletion; it never reads the verification workspace.
317:         if evaluator.sandbox.sandboxes:
318:             raise RuntimeError("Cannot replace an already active sandbox manager.")
````

## s010

来源：`reports/trajectory_cycle01_pair_r02_review_20261115/OVERVIEW.md:24`

````text
24: 两臂phase/cleanup闭合、not_run为空；包装器exit0，实际main python_exit1表示批内未解。全部请求使用同adapter别名、六工具及真实系统提示/采样核对通过，无落回底座证据。已返回可解析摘要167/141全部进入后续主请求；不能归为compact普遍丢失。无本轮OOM/IPC中断。
25: 
26: ## 值得保留与需要纠正
27: 
28: 1. **保留调用/恢复改善，补语义闭环。** 7205/T02的`if any(_netrc)`破坏None路径；5624把兄弟scope混合；7328删除中间history。这些需要旧行为和新行为一起成立的目标，单次“编辑成功”回执不足。
29: 2. **把已知证据落实为源码。** 12942/T02到第29次才编辑，额度已用完；9425/T1、14430/T1已见关键事实仍没有修复。同题另一臂能做，不代表当前状态已会稳定采取该动作。
30: 3. **学会选择能击中问题的检查。** 11194/T1只测UploadFile，遗漏bytes转换；3521/T02多做均匀字符探针，没覆盖混宽；3468回避原pipe触发，普通输出通过不能说明修好。
31: 4. **JSON合法不等于操作正确。** 15800/T1的sed `w`本想读源码却清空4694行；9753/T1通过调用用户函数来分类，真实运行错误后撤回合理。后者应学习正确替代，而不是硬保留已证明有错的实现。
32: 5. **不强行补未知原因。** 13713/T1已修改完整公开调用链，但正式仍missing且自测未运行，确切剩余根因未知；3052返回规范化、14791输出结构都须从公开可见来源建立期望，不能把裁判后验灌进教师。
````

## s011

来源：`reports/trajectory_cycle02_pair_r01_review_20261116/overview.md:14`

````text
14: 主要信号：低温重复相同搜索并忽略预算拒绝、空补丁和失实终答；高温产生补丁却修错接口/写坏语法；少数响应在文本/推理里写伪工具标记而未执行。39摘要均送达，但个别内容对编辑状态判断有误。不存在“工具提醒/摘要完全没送达”的解释。D04相关11题188目标的有限核查未找到坏短句直接复制，且有正确编辑目标；不能据此证明整个监督混合最优。
15: 
16: 记录523调用尝试＝447计费+32免费status+15免费submit+29额度拒绝；581HTTP元数据/路由已核，动作参数与ADK及16正式audit回执逐值绑定。可见reasoning全文深读8条T02与T1/11194，其余295条明确UNKNOWN/masked；未假称全文读完所有重复前缀或重审全部2090训练行。
17: 
18: 简报校正：14077仅T02通过，T1未运行；“精确172秒”是相近墙钟值，不是共同超时；13537才是两臂都过。恢复后原表确认GPU2 PID<OPERATION_ID>、GPU3 PID<OPERATION_ID>，UID987/目标UUID/脚本/驻留内存/holder日志一致；<OPERATION_ID>无原件支持。两臂safe_to_pack/无残余自有进程闭合，保持原STOPPED/failed状态，不回写为completed。
19: 
20: 本地判断：接纳这一份真实的部分评测证据；C02暂不自动晋级，不从本轮推断全129或隐藏集分数。训练产物有效性接纳D090保持，但有限能力测试出现回退，后续应采用短训练、checkpoint和单变量行为对照。
21: 
22: [独立结论](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/review.md) · [逐题表](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/cases.md) · [机器结论](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/review.json) · [训练方法与最新公开源码调研](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/lora_method_reassessment_20261116/overview.md) · [待讨论实验](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/lora_method_reassessment_20261116/PROPOSED_NEXT_EXPERIMENTS.md)。没有新训练、模型/SDK/工具运行、采集、服务器或Kaggle操作，无新增hash，提示和预算保持。
````

## s012

来源：`runtime/request_log.py:1`

````text
1: """Passive localhost OpenAI relay for local diagnostics, not submission code.
2: 
3: Bodies are forwarded unchanged. ``response.raw`` contains HTTP entity bytes
4: (including SSE), after HTTP transfer framing, not a tokenizer/internal trace.
5: No HTTP headers are recorded. Call ``check()`` at task boundaries; a normal
6: context-manager exit also checks recorder integrity. HTTP/API failures and
7: cancelled or interrupted requests belong to the official evaluator's task
8: lifecycle, not this recorder's run-level policy. An unwritable disk can prevent
9: a terminal metadata update, but that recorder failure remains sticky in memory.
````

## s013

来源：`runtime/evidence.py:1`

````text
1: """Local observation of official events and Agent-workspace cleanup.
2: 
3: These helpers neither run an Agent loop nor change official tools or scoring.
4: Snapshots are bounded, non-atomic observations, never submission predictions.
5: """
6: 
7: import json
8: import os
9: import re
````

## s014

来源：`reports/trajectory_cycle01_pair_r02_review_20261115/OVERVIEW.md:5`

````text
5: 一个干净GPT-6.1 Sol/max完成语义独审，root核输入、统计和回传内容。所有258题有处置，6464原始工具动作与ADK/ATIF逐值绑定，7031请求核配置与路由。重点深审20温度分歧及共同丢解14485；不声称逐字阅读所有重复HTTP前缀、所有长stdout及全部reasoning。完整范围与逐题证据见[独审报告](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/report.md)、[逐题案例](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/cases.md)。
6: 
7: ## 结果与可比性
8: 
9: |指标|原r04无adapter T0.2|同r04+LoRA T0.2|同r04+LoRA T1.0|
10: |---|---:|---:|---:|
11: |resolved/129|51|48|52|
12: |缺必填字段的调用/涉及题|107/16|1/1|8/6|
13: |实际会话截止|5|1|1|
````

## s015

来源：`docs/BASELINE.md:155`

````text
155: [数据计划](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/experiments/trajectory_dataset_r01/README.md)和[34项核验](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/reports/trajectory_training_sampling_20261112/verification.json)仅检查129分区/任务槽/预声明触发/源包直字节及无掉题、无限制题误入正例的反例，不是新SDK/模型/训练结果。129开发评测保留，115候选/2空基线/12已知限制复用原控制，不重新启动CPU校准链。r04基线尚无结果；r03审查只用于选题，缺真实中间工作区不能继续旧会话。当前新采集0、可训练token导出0、已有两个原型不自动晋级。
156: 
157: 接手合同要求实际请求边界/工作区/工具状态/剩余预算一致；终局patch或摘要不能替代现场。窗口只监督当前目标，前缀中的先前正确教师动作也mask，避免重复训练。真实摘要触发不要求已有源码编辑，以覆盖未写入的错误进展摘要。后续采集实现及native模板标签验收另做，当前冻结发行和GPU2单臂交接不受影响。
158: 
159: ## 2026-11-12　r04-T02训练前基线交接：GPU2单臂
160: 
161: D064原SELF-v17-r04-T02 6842B ZIP整字节进入新交接，四成员与冻结件相等。继承r03已验证A800执行器/官方判分，单臂只允许t02，129序/任务资源/293包/98官方源及已知117/115/12校准口径保持；无新模型成绩。新run_single保存真实main退出、monitor停止和owned_server_cleanup证明；完整exit1允许，部分/源漂移/清理未闭合不产生正式baseline_record。CPU只短来源与实际SDK声明核对，不扩完整门禁。
162: 
163: 用户最终GPU2而非前案GPU3或旧GPU5。新device_binding只读核物理index2/PCI65:00.0/A80080G并保存真实UUID，后续复核不可改绑；原vLLM UUID→物理index适配和逻辑CUDA0的UUID验证保持。两次空闲快照/2秒GPU与60秒磁盘记录，3/5号不操作。预留按实际状态PRESERVED/RESTORED/RESERVED_NEW/NOT_STARTED_EMPTY/NOT_RESTORED如实记录，不能把原空闲说成恢复服务。
````

## s016

来源：`learning_pipeline/evaluation/panel_package.py:58`

````text
58:     # Verify the entire original calibration; project its descriptive view only afterwards.
59:     def gate(s):
60:         a=s.index('def verify_reference_binding()');b=s.index('def require_acceptance()',a)
61:         block=s[a:b].replace('settings.TASK_IDS','settings.CALIBRATION_TASK_IDS')
62:         block=replace(block,'    return coverage','    require(set(settings.TASK_IDS) <= set(coverage["target_discriminated_ids"]), "Panel includes a non-discriminating task")\n    return coverage')
63:         return s[:a]+block+s[b:]
64:     edit('runtime/alignment_gate.py',gate)
65:     edit('check_resources.py',lambda s:s.replace('!=settings.TASK_IDS','!=settings.CALIBRATION_TASK_IDS'))
66:     edit('runtime/device_binding.py',lambda s:replace(s,'INDEX = {"t02":2,"t10":3}[os.environ.get("HARNESS_K_VARIANT","t02")]',
````

## s017

来源：`docs/BASELINE.md:131`

````text
131: 原129/51与78未解不变，12校准限制单列。66非限制未解中A调用/编辑表示24、B动作推进/恢复/验证50、C实现语义覆盖22；全78分别28/51/22，可重叠且不是主要失败原因占比。严格SDK缺参/范围拒绝仅9/78，宽A还含静默字段错误、路径及编辑表示。C另44/66不确定；known-unfinished不等同理解错误。详细覆盖是完整动作序列索引和分段材料、关键原行，不是全HTTP/thinking全文。
132: 
133: root核129计费与summary相等、3666原HTTP可见调用与ADK参数一致、1063 budget_warning全部计数正确且进入后续主请求；正式/输出四成员与活动提示绑定。没有parser前原始文本，不证明格式错误只能由模型引起。58未解用满28，其中45在Agent300秒内；成功题也有14用满，不能单凭额度判流程错。[统一报告](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/reports/self_v17_r04_failure_taxonomy_20261113/root_review.zh-CN.txt)与[统计核验](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/reports/self_v17_r04_failure_taxonomy_20261113/root_summary.js … [长行省略]
134: 
135: ## 2026-11-13　r04基线已完成，训练数据未放行
136: 
137: 原r04-T02服务器129/51、phase/cleanup completed、not_run空，main exit1为完整批次含未解题；驱动29743.89s，Agent合计27842.71s。input/output与本地正式6842B四成员一致；用户线上截图.13，线上上传ZIP未下载。对r03 T02 52保持47/新4/丢5。[统计与绑定](../reports/trajectory_correction_r01_review_20261113/baseline_check.json)保存原口径，不推同题单轮为因果或预测隐藏分。
138: 
139: 数据包11训练行全部submit，238正标签含128 thought渲染token及110动作/handoff（含标记）；root和单Terra独立原生重编码相同，无编辑监督，0经验证教师纠正。5作业真实重放与官方时钟存在，但主要在教师队列等待时到截止，并非教师完成动作后解题失败。378项验收不覆盖动作mask与语义资格。[本地不放行记录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/reports/trajectory_correction_r01_review_20261113/training_gate.json)不改原包eligibility。仅只读数据/tokenizer检查，无新工具、裁判、模型或训练；下一采集方式与预算尚待明确，旧交接不重入。
````

## s018

来源：`reports/trajectory_cycle01_pair_r02_review_20261115/OVERVIEW.md:26`

````text
26: ## 值得保留与需要纠正
27: 
28: 1. **保留调用/恢复改善，补语义闭环。** 7205/T02的`if any(_netrc)`破坏None路径；5624把兄弟scope混合；7328删除中间history。这些需要旧行为和新行为一起成立的目标，单次“编辑成功”回执不足。
29: 2. **把已知证据落实为源码。** 12942/T02到第29次才编辑，额度已用完；9425/T1、14430/T1已见关键事实仍没有修复。同题另一臂能做，不代表当前状态已会稳定采取该动作。
30: 3. **学会选择能击中问题的检查。** 11194/T1只测UploadFile，遗漏bytes转换；3521/T02多做均匀字符探针，没覆盖混宽；3468回避原pipe触发，普通输出通过不能说明修好。
31: 4. **JSON合法不等于操作正确。** 15800/T1的sed `w`本想读源码却清空4694行；9753/T1通过调用用户函数来分类，真实运行错误后撤回合理。后者应学习正确替代，而不是硬保留已证明有错的实现。
32: 5. **不强行补未知原因。** 13713/T1已修改完整公开调用链，但正式仍missing且自测未运行，确切剩余根因未知；3052返回规范化、14791输出结构都须从公开可见来源建立期望，不能把裁判后验灌进教师。
33: 
34: D03并非没有源码学习：已有正确源码目标的12942/14485/5077/5624仍失分；同时，旧通过7205/9753/7427的最终核心代码因保护/来源/整call资格并未全进正目标。应按“已教仍失效”和“核心目标未教”分别处理，不能只用1990行或129题覆盖数判断训练充分。
````

## s019

来源：`learning_pipeline/collection/runner.py:69`

````text
69: def bind_execution(job,index,stage,receipt,previous,actual,responses):
70:     """Bind the last added proposal to actual SDK consumption and tool receipts."""
71:     teacher=job/f'teacher_{index:03d}';entry=responses[-1]
72:     request_index=len(responses)
73:     if receipt['replayed_responses']!=request_index:raise RuntimeError('Latest teacher response was not consumed by official replay')
74:     comparison=read(stage/'request_comparisons'/f'{request_index:04d}.json')
75:     state=read(stage/'state_comparisons'/f'{request_index:04d}.json')
76:     if not comparison['equal_under_declared_comparison'] or not state['equal']:
77:         raise RuntimeError('Teacher proposal input/state binding failed')
````

## s020

来源：`learning_pipeline/collection/runner.py:252`

````text
252:             req=read(stage/'next_request.json')
253:             if not (stage/'task_state.zip').is_file() or not (stage/'task_state.json').is_file() or not req.get('task_state'):
254:                 raise RuntimeError('Captured boundary lacks complete task-file state evidence')
255:             if idx==0:
256:                 comparison=compare_requests(original_cut,req['openai_request'])
257:                 write_new(stage/'original_cut_request_comparison.json',comparison)
258:                 if not comparison['equal_under_declared_comparison']:raise RuntimeError('Original cut request drift')
259:                 presentation_variant|=comparison['presentation_order_variant']
260:             if stop_fragment:
````

## s021

来源：`learning_pipeline/collection/equivalence.py:168`

````text
168: def compare_receipts(tool,a,b,args=None):
169:     equal=receipt(tool,a,args)==receipt(tool,b,args)
170:     ga=_presentation_groups(a,args) if tool=='run_command' else None
171:     gb=_presentation_groups(b,args) if tool=='run_command' else None
172:     variant=equal and ga is not None and gb is not None and ga==gb and a['stdout']!=b['stdout']
173:     return {'raw_equal':a==b,'equal_under_declared_comparison':equal,'comparison_version':COMPARISON_VERSION,
174:         'presentation_order_variant':variant,
175:         'reconstruction_condition':'RECONSTRUCTED_PRESENTATION_ORDER_VARIANT' if variant else 'OFFLINE_RECONSTRUCTION_DECLARED_COMPARISON',
176:         'comparison_rules':RULES,'not_a_byte_identical_receipt_claim':a!=b}
````

## s022

来源：`learning_pipeline/contracts/teacher_fragment_review.md:9`

````text
9: ## 先核真实执行
10: 
11: - 只接收 `job_result.status=FRAGMENT_CAPTURED`、`collection_mode=fragment`、`overall_verification=NOT_RUN`。`final_stage` 指最后真实下一请求边界，不是整题终态。该链没有新 TaskResult、独立判分或虚构 submit/final。
12: - `teacher_000` 到末项全部逐项处置，包括中途实际 summary；编号完整且唯一。每个实际提案须有 ACCEPTED 的 `response.json`、原 `proposal.txt`、`response.raw.json`、实际 `teacher_input.txt`、无 native 工具的 CLI 事件，以及下一 stage 的实际请求/状态比较和 ADK 调用及后续回执。`target_binding` 的执行计划不能代替 `execution_binding.status=EXECUTED`。
13: - 所有 stage（包括前缀 stage_000 和末 stage）必须真实 BOUNDARY_CAPTURED、cleanup 完成、`protected_mutation_observed=false`、cleanup 前保护字节相等、无 audit_error，并保留下一请求与 task_state.zip/json。首次原学生工作区未观测仍记 RECONSTRUCTED，不称严格同现场。
14: - 源码写入、验证或捕获结论、正 reasoning、保护/历史访问边界，须由独立审查者或执行协调员实际核实。不得因工具 ok 就认可未知代码。保护污染、历史/私有证据访问、未执行或工具拒绝不能正训。
15: - 非零检查仅在它确实是有效失败诊断时可接纳：lane 必须 VALID_DIAGNOSTIC，另写 `receipt_assessment=EXPECTED_DIAGNOSTIC_FAILURE`，并解释真实失败揭示的缺口。畸形脚本、错误断言或缺参拒绝不属于此例外。
16: 
17: ## 逐目标资格
````

## s023

来源：`learning_pipeline/contracts/teacher_natural_chain_review.md:7`

````text
7: - `overall_verification=SEE_OFFICIAL_TASK_RESULT`；所有实际proposal（包括summary和最后真实text）逐一处置，不为达到计划步数伪造工具动作。
8: - 中间stage保持真实 BOUNDARY_CAPTURED、next_request、task_state及实际请求/状态比较；最终stage必须真实 COMPLETED、task_result_created=true、对应真实boolean resolved，patch/verification与TaskResult直接字节相等。终态没有task_state.zip时，不补造或声称已捕获。
9: - 仍逐proposal核ACCEPTED payload/实际CLI事件/原始公开说明/实际teacher_input/下一stage请求与状态比较/唯一工具调用与后续回执。最后text另外须有REPLAYED_VISIBLE_TEXT绑定，且和最终ADK最后可见model文本一致。
10: - action-mode请求数可能包括真实text终答。分别记录实际工具数、真实终答数、summary数；不能把6个action-mode提案称6个工具动作。
11: - 动作lane可为LOCAL_ACTION_VERIFIED、VALID_DIAGNOSTIC；只有真实TaskResult.resolved=true且审核确认语义/范围的动作才能FULL_REPAIR_VERIFIED。官方通过不能代替逐动作语义审核。未解链仍可有有依据的局部目标。
12: - `final_approved=true`仅限实际最后一个action模式text，action_approved/reasoning_approved必须false，并提供 `qualification.final` 的独立或执行协调员来源。未出现终答不造target；summary不当终答。final的lane可null，表示事实终答资格；FULL字样仍要求resolved=true。
13: 
14: 示例终答target：
15: 
````

## s024

来源：`learning_pipeline/collection/cli_worker.py:27`

````text
27: For mode=action, decision_summary may contain a concise public decision explanation:
28: the already observed evidence and the next intended action/check, usually a few sentences.
29: Give no private chain-of-thought. Do not invent observations, future results or an answer key.
30: This field is explicitly authored training material, not a transcript of internal reasoning.
31: Use an empty decision_summary for compaction summaries and when no explanation is useful.
32: For kind=text use name="", arguments={} and the actual text. For kind=tool_call use
33: text="", the declared tool name and its native JSON arguments object. Do not JSON-encode
34: that object a second time. Supply the nullable line bounds and allow_multiple explicitly.
35: Return only this object. Do not provide private chain-of-thought, an explanation of these
````

## s025

来源：`learning_pipeline/native.py:107`

````text
107:     def complete_example(self,sample_id,request,target,selected_call_ids,metadata,reasoning=False,final_text=False):
108:         """Supervise the complete *current* accepted assistant completion.
109: 
110:         This is deliberately separate from ``example``: old action/joint masks
111:         keep their exact behaviour.  A no-reasoning derivative must remove its
112:         current thought explicitly before calling this method; retaining an
113:         unknown thought as masked answer context is not a completion example.
114:         Every real call in a multi-call completion must be qualified.
115:         """
````

## s026

来源：`learning_pipeline/native.py:32`

````text
32: class Native:
33:     def __init__(self,model=MODEL):
34:         self.model=Path(model);self.tokenizer=Tokenizer.from_file(str(self.model/'tokenizer.json'))
35:         env=ImmutableSandboxedEnvironment(trim_blocks=True,lstrip_blocks=True,extensions=['jinja2.ext.loopcontrols','jinja2.ext.do'])
36:         def fail(s):raise TemplateError(s)
37:         env.globals.update(raise_exception=fail,strftime_now=lambda fmt:datetime.now().strftime(fmt))
38:         env.filters['tojson']=lambda x,ensure_ascii=False,indent=None,separators=None,sort_keys=False:json.dumps(x,ensure_ascii=ensure_ascii,indent=indent,separators=separators,sort_keys=sort_keys)
39:         self.template=env.from_string((self.model/'chat_template.jinja').read_text(encoding='utf-8'))
40:     def render(self,messages,tools,generation):
````

## s027

来源：`learning_pipeline/native.py:116`

````text
116:         calls=target.get('tool_calls') or []
117:         if ({c['id'] for c in calls}!=set(selected_call_ids) or len({c['id'] for c in calls})!=len(calls)
118:                 or len(selected_call_ids)!=len(calls)):
119:             raise ValueError('Complete completion requires every real call to be selected exactly once')
120:         if not reasoning and any(target.get(k) for k in ('reasoning','reasoning_content')):
121:             raise ValueError('Unsupervised current reasoning must be explicitly removed from completion derivative')
122:         row=self.example(sample_id,request,target,selected_call_ids,metadata,reasoning=reasoning,final_text=final_text)
123:         prompt_tokens=row['metadata']['prompt_tokens'];ids=row['input_ids']
124:         reason=row['labels_reasoning']
````

## s028

来源：`reports/fresh_lora_implementation_20261116/data_builder/implementation.md:63`

````text
63: ## 实际 A 中后回合修复差量
64: 
65: root 第一次实际导出保留了 953 行/154 task，但全部 37 个 A 目标因 `A_ORIGINAL_PREFIX_NOT_EXACTLY_RECONSTRUCTED` 被隔离。该同型全层隔离是构建器遗漏源边界分支，不能解释为 37 个目标语义不合格；首次导出未 audit/freeze，root 保全为 attempt1 后重新导出。
66: 
67: 根因是作者 A 源模板的工具回执后 generation prefix 含 `<|channel>thought\n`，长 18 字符；本地当前 r04 归档模板把该入口留给 completion。作者 `render_example` 原码明确以 `prefix.endswith(FORCED)` 选择两种 completion 边界；首次实现仅测初始 index0，漏测工具回执后的分支。原历史本体没有差异。例如 `chatcmpl-fef133a1decb95466f12dc8ca6ec7a5e` 的 index6，当前模板 prefix 长 24410、源保存 24428；index7 为 25406/25424；index1 … [长行省略]
68: 
69: 修复按实际历史末消息 `role=tool`、且规范渲染必须严格以 `<tool_response|>` 结束时，确定性重建源 generation 的 FORCED 入口；没有从任意差异里猜测或忽略内容。之后仍要求完整 source prefix 逐字节相等，按作者原分支构建完整 completion 并核 IDs，另增加 prompt/completion/context token 数全相等检查。当前训练派生仍用真实本地模板渲染，没有往训练输入人工追加源 FORCED 入口。
70: 
71: `check_a_multiturn.py` 对全部 **14 条 A/382 个目标** 重建通过：14 初始入口、368 工具回执后入口；源 prefix、completion、completion IDs、三种 token 数均精确相等。两个 curator selection 的 **37/37 A 目标** 全部重建通过。完整逐目标结果在 `a_multiturn_reconstruction.json`，失败列表为空。此为机械渲染绑定，不是重新语义审查或 SDK 重执行。
````

## s029

来源：`reports/fresh_lora_implementation_20261116/overview.md:13`

````text
13: ## 最终数据不是原2090行重复训练
14: 
15: | 来源 | 目标回合 | 任务ID | 实际目标系数质量 |
16: |---|---:|---:|---:|
17: | 本地Gemma合格行为（含选中旧来源） |245|65|27.9219% |
18: | 本地Sol纠偏（含选中旧来源） |475|76|42.7031% |
19: | 公开Native |233|41|25.6250% |
20: | 公开A转换来源 |37|6|3.7500% |
21: | 合计 |**990**|**160**|**100%** |
````

## s030

来源：`learning_pipeline/fresh.py:226`

````text
226: def _task_stage_weights(groups):
227:     """Uniform rows are traversed once, hence m=N/(T*K*n), never p alone."""
228:     counts=defaultdict(Counter)
229:     for task,stage in groups:counts[task][stage]+=1
230:     total=len(groups);tasks=len(counts)
231:     if not total:raise ValueError('No qualified fresh completion rows')
232:     for task,stages in counts.items():
233:         if not set(stages)&{'edit','verify','recover_finish'}:
234:             raise ValueError('Read-only task reached main-lane weighting: '+task)
````

## s031

来源：`learning_pipeline/training/train_adapter.py:112`

````text
112:     base_cfg=AutoConfig.from_pretrained(a.model,local_files_only=True)
113:     qdict=base_cfg.quantization_config
114:     if qdict.get('quant_method')!='compressed-tensors':raise RuntimeError('Expected archived packed QAT base')
115:     # Pinned Transformers 5.13.1 uses run_compressed=False, not the later
116:     # dequantize=True API. Restore existing packed values, no NF4 re-quantization.
117:     quant=CompressedTensorsConfig.from_dict(qdict);quant.run_compressed=False
118:     base,loading=Gemma4ForConditionalGeneration.from_pretrained(a.model,local_files_only=True,
119:         quantization_config=quant,dtype=torch.bfloat16,device_map={'':local_rank},attn_implementation='action_chunked',output_loading_info=True)
120:     loading={k:sorted(v,key=repr) if isinstance(v,set) else v for k,v in loading.items()}
````

## s032

来源：`learning_pipeline/training/train_adapter.py:134`

````text
134:     pattern=r'.*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$'
135:     targets=[n for n,m in base.named_modules() if re.fullmatch(pattern,n) and isinstance(m,torch.nn.Linear)]
136:     if not targets or any('vision' in n or 'audio' in n for n in targets):raise RuntimeError('Invalid text-only LoRA targets')
137:     text_cfg=base.config.get_text_config()
138:     expected_count=sum(6 if typ=='full_attention' and text_cfg.attention_k_eq_v else 7 for typ in text_cfg.layer_types)
139:     if text_cfg.num_hidden_layers!=60 or expected_count!=410 or len(targets)!=expected_count:
140:         raise RuntimeError('Expected exactly 410 existing text projections across 60 QAT-31B layers')
141:     for idx,layer in enumerate(base.model.language_model.layers):
142:         for branch,names in [('self_attn',['q_proj','k_proj','v_proj','o_proj']),('mlp',['gate_proj','up_proj','down_proj'])]:
````

## s033

来源：`learning_pipeline/training/fsdp_runtime.py:18`

````text
18: def shard_frozen_decoders(objective, base, trainable, *, device, expected_layers=60):
19:     from torch.distributed.device_mesh import init_device_mesh
20:     from torch.distributed.fsdp import fully_shard
21:     from torch.distributed.tensor import DTensor
22: 
23:     layers=base.model.language_model.layers
24:     if len(layers)!=expected_layers:raise ValueError('Unexpected decoder layer count')
25:     names=[(n,list(p.shape),str(p.dtype)) for n,p in trainable]
26:     structure={'adapters':names,'base_parameters':parameter_inventory(base)}
````

## s034

来源：`learning_pipeline/training/fsdp_runtime.py:88`

````text
88: def average_adapter_gradients(trainable, bucket_bytes=16*1024**2):
89:     """One collective validation before bounded, deterministically ordered sums."""
90:     device=trainable[0][1].device
91:     invalid=torch.zeros((),dtype=torch.int32,device=device)
92:     for _,p in trainable:
93:         if p.grad is None:invalid.add_(1)
94:         else:invalid.add_((~torch.isfinite(p.grad).all()).to(torch.int32))
95:     dist.all_reduce(invalid,op=dist.ReduceOp.MAX)
96:     if int(invalid):raise RuntimeError('Missing/nonfinite LoRA gradient on at least one rank')
````

## s035

来源：`learning_pipeline/training/fsdp_runtime.py:34`

````text
34:     # Restored compressed-tensors Linear modules can retain integer Parameters
35:     # such as weight_shape. Torch 2.10 FSDP2 first wraps them with requires_grad
36:     # defaulting to True, even though they were frozen. Preserve, do not cast or
37:     # delete this state. Floating restored weights and scales are still sharded.
38:     nonfloat={n:p for n,p in layers.named_parameters() if not p.is_floating_point()}
39:     if any(p.requires_grad for p in nonfloat.values()):raise ValueError('Nonfloating trainable state is not supported')
40:     auxiliary_bytes=sum(p.numel()*p.element_size() for p in nonfloat.values())
41:     auxiliary=[{'name':n,'shape':list(p.shape),'dtype':str(p.dtype),'bytes':p.numel()*p.element_size()} for n,p in nonfloat.items()]
42:     with torch.no_grad():
````

## s036

来源：`learning_pipeline/training/train_core.py:13`

````text
13: def _chunked_mlp_forward(self, hidden_states):
14:     """Chunk the position-wise MLP, not attention or the causal input sequence."""
15:     if hidden_states.ndim!=3 or hidden_states.shape[1]==0:
16:         raise ValueError('MLP chunking expects a nonempty batch/sequence/hidden tensor')
17:     pieces=[]
18:     for block in hidden_states.split(self._trajectory_chunk_tokens,dim=1):
19:         original=self._trajectory_original_forward
20:         if torch.is_grad_enabled():
21:             output=checkpoint(original,block,use_reentrant=False,preserve_rng_state=True)
````

## s037

来源：`learning_pipeline/training/train_core.py:223`

````text
223: def chunked_attention(module,query,key,value,attention_mask,dropout=0.0,scaling=None,sliding_window=None,**kwargs):
224:     """Training-only batch1 causal text attention, including Gemma local windows.
225: 
226:     No caches/padding/multimodal masks are accepted. Key/value GQA is explicit.
227:     Query chunks are recomputed for backward, without an L x L attention matrix.
228:     """
229:     if query.shape[0]!=1 or query.shape[-2]!=key.shape[-2] or key.shape[-2]!=value.shape[-2]:
230:         raise ValueError('Only unpadded batch1 full-sequence training is supported')
231:     if dropout:raise ValueError('This recipe fixes attention dropout to zero')
````

## s038

来源：`learning_pipeline/training/train_core.py:276`

````text
276: def selected_action_loss(model,input_ids,labels,head_chunk=64,token_weights=None,*,diagnostic_masks=None):
277:     """Compute exact causal CE on supplied targets only, without full-vocab L logits."""
278:     if input_ids.ndim!=2 or input_ids.shape[0]!=1 or labels.shape!=input_ids.shape:
279:         raise ValueError('Microbatch must be one intact sequence')
280:     if int(labels[0,0])!=-100:raise ValueError('First token cannot be a causal target')
281:     positions=torch.nonzero(labels[0]!=-100,as_tuple=False).flatten()
282:     if positions.numel()==0:raise ValueError('No action labels')
283:     base=model.get_base_model() if hasattr(model,'get_base_model') else model
284:     output=base.model(input_ids=input_ids,use_cache=False,return_dict=True)
````

## s039

来源：`learning_pipeline/training/SOURCES.md:7`

````text
7: ## 可核作者源码
8: 
9: - [Scott Weeden 的公开训练kit](https://www.kaggle.com/datasets/scottweeden/gemma-4-developer-agent-tool-lora-kit)：本地归档为 `vendor/lora_research_20261112/scott_kit/4/scripts/train.py`（43326字节）及 `0/recipe/gemma4_31b_peft_recipe.md`（6607字节）；原始来源和许可记录见该目录外的 `SOURCE_NOTES.txt`。实际代码提供了按query分块的checkpoint attention、仅目标位置的chunked CE、 … [长行省略]
10: - 上述作者实际脚本还使用NF4，并在注意力注册失败时将长度降至6144、过滤长行；这些行为不进入本框架。我们保留完整32768上限，任何超长行或注册失败都显式失败。原PEFT adapter保持不变，按实际competition wheel另做K→V服务导出，不套作者宽后缀配置。
11: - 作者榜上最佳成绩没有绑定到这一份adapter/脚本，不作为本包效果证据。其他已归档教程存在Gemma2/随机fallback、模拟训练或全token标签问题，未作为运行实现采用。
12: 
13: ## 第一方版本依据
14: 
15: - [Transformers 5.13.1 Gemma4接口](https://huggingface.co/docs/transformers/v5.13.1/en/model_doc/gemma4)：核对实际 `Gemma4ForConditionalGeneration`、原生工具模板与文本配置；不把普通31B或新版接口替代现有QAT父模型。
````

## s040

来源：`learning_pipeline/training/train_adapter.py:300`

````text
300:                     if micro>=len(local_batch):
301:                         communication_only_backward(objective,torch.device('cuda',local_rank))
302:                         continue
303:                     i=local_batch[micro]
304:                     row=rows[i];weight=row_weights[i]
305:                     sync=objective.no_sync() if world>1 and not fsdp and micro<len(local_batch)-1 else nullcontext()
306:                     with sync:
307:                         with torch.autocast('cuda',dtype=torch.bfloat16):
308:                             result=action_loss(row,return_diagnostics=fresh)
````

## s041

来源：`learning_pipeline/training/fsdp_runtime.py:114`

````text
114: def communication_only_backward(objective,device,mixed=True):
115:     """Equal FSDP slots for an uneven tail; no data row/loss/RNG contribution."""
116:     devices=[device.index] if device.type=='cuda' else []
117:     with torch.random.fork_rng(devices=devices):
118:         ids=torch.zeros((1,2),dtype=torch.long,device=device)
119:         with torch.autocast(device.type,dtype=torch.bfloat16,enabled=mixed):
120:             zero=objective(ids,None,communication_only=True)
121:         zero.backward()
122: 
````

## s042

来源：`learning_pipeline/training/train_core.py:324`

````text
324: def selected_completion_loss(model,input_ids,labels_action,labels_reasoning,head_chunk=64,*,return_diagnostics=False):
325:     """One row's mean CE across every qualified current-completion target."""
326:     if labels_action.shape!=input_ids.shape or labels_reasoning.shape!=input_ids.shape:
327:         raise ValueError('Bad completion component dimensions')
328:     action=labels_action!=-100;reason=labels_reasoning!=-100
329:     if (action&reason).any() or not action.any():raise ValueError('Need disjoint completion targets with an accepted action/final')
330:     labels=torch.where(action,labels_action,labels_reasoning)
331:     if not return_diagnostics:return selected_action_loss(model,input_ids,labels,head_chunk)
332:     loss,sums=selected_action_loss(model,input_ids,labels,head_chunk,diagnostic_masks=(action,reason))
````

## s043

来源：`learning_pipeline/server/export_serving.py:16`

````text
16: def expand(tensors,config,text_config):
17:     if len(tensors)!=820 or config['r']!=16 or config['bias']!='none' or config.get('modules_to_save'):
18:         raise ValueError('Unexpected training adapter contract')
19:     global_layers=[i for i,typ in enumerate(text_config['layer_types']) if typ=='full_attention']
20:     if not text_config['attention_k_eq_v'] or global_layers!=list(range(5,60,6)):raise ValueError('Unexpected shared K/V architecture')
21:     wanted=set()
22:     for i in range(60):
23:         for branch,projs in [('self_attn',['q_proj','k_proj','o_proj']+([] if i in global_layers else ['v_proj'])),('mlp',['gate_proj','up_proj','down_proj'])]:
24:             for proj in projs:
````

## s044

来源：`reports/trajectory_cycle02_fsdp4_r01_return_review_20261116/overview.md:7`

````text
7: | 执行链 | 九退出码全0，524更新；每轮2090条各一次，合4180次真实消费 | 完整按配方执行，尾2没有补样/丢样 |
8: | 父权重 | 原820 FP32在同步前后精确继承，新AdamW；最终820/820张量均变化且有限 | 续训真实生效，父件未覆盖 |
9: | 训练损失 | 第1/2轮按真实样本平均0.226930→0.192685，下降15.09% | 对D04训练目标拟合改善；不是验证loss |
10: | 最后20步 | 按真实样本平均0.173166（每步等权平均0.166827） | 比末尾单批更有代表性，仍不能代表解题能力 |
11: | 耗时 | 训练loop13.594h，worker13.669h；四卡总971.6输入token/s | 比C01的10.295h更长，工作量与长度也变大 |
12: | 显存 | FSDP ready后最长+训练allocated峰52.7096GiB；最长reserved59.45–62.85GiB，估算余量15.70–19.10GiB | 这批26979完整长度稳定承载；52.7不是全生命周期总显存峰值 |
13: | 导出/服务 | 820→840 BF16逐值正确，195042419B六成员单adapter ZIP，真实vLLM短加载通过 | 可作为评测候选；短探针不等于解题提分 |
14: 
15: ## 需要纠正的loss口径
````

## s045

来源：`reports/trajectory_cycle01_fsdp4_r05_return_review_20261114/README.md:5`

````text
5: | 项目 | 原件核验结果 |
6: |---|---|
7: | 回传 | 1,038,477,616 B；186 regular=185 payload+summary，安全解包/直接字节归档 |
8: | 训练 | 498正式更新；D03 1990×2，全部3980样本访问符合冻结洗牌/四rank分配 |
9: | 耗时 | 训练循环37062.58秒，约10.30小时 |
10: | loss | 首步0.22633、末步0.05348；前20步均值0.36723、末20步0.09394。不是简报中0.54→0.29口径 |
11: | 显存 | 完整19356-token两累积，四rank峰值allocated43.84 GiB；估算余量28.45/30.24/30.90/29.08 GiB |
12: | 权重 | 820 FP32训练张量有限，转换840 BF16服务张量逐值相符，包含20共享K/V复制 |
13: | 加载 | 最终真实vLLM加载及单token数值差异通过；只证明加载/变化，未证明解题收益 |
````

## s046

来源：`reports/trajectory_cycle02_fsdp4_r01_return_review_20261116/overview.md:5`

````text
5: | 项目 | 原件核对结果 | 评价 |
6: | --- | --- | --- |
7: | 执行链 | 九退出码全0，524更新；每轮2090条各一次，合4180次真实消费 | 完整按配方执行，尾2没有补样/丢样 |
8: | 父权重 | 原820 FP32在同步前后精确继承，新AdamW；最终820/820张量均变化且有限 | 续训真实生效，父件未覆盖 |
9: | 训练损失 | 第1/2轮按真实样本平均0.226930→0.192685，下降15.09% | 对D04训练目标拟合改善；不是验证loss |
10: | 最后20步 | 按真实样本平均0.173166（每步等权平均0.166827） | 比末尾单批更有代表性，仍不能代表解题能力 |
11: | 耗时 | 训练loop13.594h，worker13.669h；四卡总971.6输入token/s | 比C01的10.295h更长，工作量与长度也变大 |
12: | 显存 | FSDP ready后最长+训练allocated峰52.7096GiB；最长reserved59.45–62.85GiB，估算余量15.70–19.10GiB | 这批26979完整长度稳定承载；52.7不是全生命周期总显存峰值 |
13: | 导出/服务 | 820→840 BF16逐值正确，195042419B六成员单adapter ZIP，真实vLLM短加载通过 | 可作为评测候选；短探针不等于解题提分 |
````

## s047

来源：`learning_pipeline/training/schedule.py:20`

````text
20: def _fresh_contract(recipe, sample_count):
21:     if type(sample_count) is not int or sample_count < 1:
22:         raise ValueError('Fresh training requires a nonempty frozen dataset')
23:     required = {'supervision': 'completion', 'initialization': FRESH_INITIALIZATION,
24:                 'optimizer_initialization': 'new_stage_empty_adamw', 'schedule': FRESH_SCHEDULE,
25:                 'rank': 16, 'alpha': 16, 'dropout': .05, 'epochs': 1,
26:                 'world_size': 4, 'microbatch_size': 1, 'gradient_accumulation': 2,
27:                 'global_batch_size': 8, 'parallelism': 'fsdp2',
28:                 'seed': 42, 'max_sequence_length': 32768, 'query_chunk': 512,
````

## s048

来源：`learning_pipeline/training/checkpoint.py:89`

````text
89: def save_final_training_state(directory, optimizer, trainable, *, step, schedule, binding,
90:                               next_cursor, rank_rng, adapter_path):
91:     """Retain one AdamW copy plus exact data cursor; no promise of usable resume."""
92:     import torch
93:     validate_finite_adapter(trainable)
94:     if step != schedule.total_optimizer_steps:raise ValueError('Final optimizer retention must be at the frozen horizon')
95:     if len(optimizer.state) != len(trainable):raise RuntimeError('Final AdamW state is incomplete')
96:     moment_bytes = 0
97:     for name, parameter in trainable:
````

## s049

来源：`reports/trajectory_cycle01_pair_r02_review_20261115/OVERVIEW.md:7`

````text
7: ## 结果与可比性
8: 
9: |指标|原r04无adapter T0.2|同r04+LoRA T0.2|同r04+LoRA T1.0|
10: |---|---:|---:|---:|
11: |resolved/129|51|48|52|
12: |缺必填字段的调用/涉及题|107/16|1/1|8/6|
13: |实际会话截止|5|1|1|
14: |连续相同工具/参数调用|81|12|12|
15: |多次submit_patch的题|30|7|4|
````

## s050

来源：`reports/fresh_lora_implementation_20261116/prior_c01_scores_20261116.json:1`

````text
1: {
2:   "date": "2026-11-16",
3:   "source": "USER_PROVIDED_SCREENSHOT",
4:   "image": "reports/fresh_lora_implementation_20261116/c01_scores_user_screenshot.png",
5:   "image_bytes": 33079,
6:   "observations": [
7:     {
8:       "description_prefix": "SELF-C01-D03-T10-r01",
9:       "status": "Succeeded",
````

## s051

来源：`reports/trajectory_cycle02_pair_r01_review_20261116/overview.md:7`

````text
7: | 臂 | 完成/计划 | resolved | C01同题同温度 | 完成题空patch | 中断活动题 | 未运行 |
8: |---|---|---:|---:|---:|---|---:|
9: | T0.2 | 10/129 | 2 | 4/10 | 6 | fastapi_14186 | 118 |
10: | T1.0 | 6/129 | 1 | 3/6 | 0 | fastapi_13786 | 122 |
11: 
12: 两臂各丢2道C01同窗成功题、0新增。T02相同前10无adapter为5/10。所有18初始消息及六工具声明与C01/base直接相等、同温度采样一致、C02路由符合输入，故不是拿旧9工具/420秒版作对照。但这是主动中断、FastAPI-only、非随机且训练已暴露的窗口，不是完整129成绩，也不足以确诊灾难性遗忘。
13: 
14: 主要信号：低温重复相同搜索并忽略预算拒绝、空补丁和失实终答；高温产生补丁却修错接口/写坏语法；少数响应在文本/推理里写伪工具标记而未执行。39摘要均送达，但个别内容对编辑状态判断有误。不存在“工具提醒/摘要完全没送达”的解释。D04相关11题188目标的有限核查未找到坏短句直接复制，且有正确编辑目标；不能据此证明整个监督混合最优。
15: 
````

## s052

来源：`reports/fresh_lora_research_20261116/overview.md:74`

````text
74: ## 下一步怎样避免又训完才发现不好
75: 
76: 先完成候选SPEC的数据视图与少量必要训练入口改造，再发一个fresh训练作业。主作业只有一次完整pass；按冻结N算U，不为了凑262或524步复制行。保存25/50/75/100%，训练结束后先用固定小面板比较底座、50%、100%权重，其他两个点按需要进入比较。最终选一个候选跑当前同配置129，保留逐题增/丢、首次有效源码修改、空patch和无信息重复；CE仅辅助。公开129早已暴露，外部留出尚未实跑时不宣称泛化提高。
77: 
78: 这是一个联合配方试验，若进步不能把功劳独归alpha或cosine。它比“换一组大参数直接再训两轮”多的是可选择的中间权重、明确的数据目标和与完整修复一致的决策标准。新训练、模型、服务器、教师采集和Kaggle均 **NOT_RUN**；[结构化汇总](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/synthesis.json)记录各主张及执行边界。
````

## s053

来源：`reports/checkpoint_panel_c03_20261116/overview.md:5`

````text
5: |GPU|候选|顺序|
6: |---|---|---|
7: |0|官方QAT底座，无LoRA|T0.2 → 清理 → T1.0|
8: |1|step62 / 50%|T0.2 → 清理 → T1.0|
9: |2|step93 / 75%|T0.2 → 清理 → T1.0|
10: |3|step124 / 100%|T0.2 → 清理 → T1.0|
11: 
12: 每臂原快照、相同12题及顺序、同提示/六工具/28-480-80/240-300/8192-4096/p.95/k64，GPU/耗时记录保留。12题都是暴露开发题；逐题差异和退化形态用于筛选，不能把一题微差或最低loss当泛化胜负。下一步选择候选再决定完整129及提交，本包不自动执行这两步。
13: 
````

## s054

来源：`learning_pipeline/__main__.py:1`

````text
1: import argparse
2: import json
3: from .core import init,make_plan,status,verify
4: from .shards import prepare as prepare_shards,inspect_reviews
5: 
6: def main():
7:     p=argparse.ArgumentParser(description='Versioned trajectory-learning workflow. Collection is explicit and requires recorded authorisation.')
8:     commands=p.add_subparsers(dest='command',required=True)
9:     commands.add_parser('init').add_argument('--config',required=True)
````

## s055

来源：`learning_pipeline/templates/stage_compact.py:16`

````text
16: def stage(folder):
17:     root=Path(folder).resolve()
18:     manifest=json.loads((root/'manifest.json').read_text(encoding='utf-8'))
19:     members=manifest['members'];names=[r['path'] for r in members]
20:     if len(names)!=len(set(names)) or len(names)!=len({n.casefold() for n in names}):
21:         raise ValueError('Duplicate member names')
22:     created=[];reused=[]
23:     with zipfile.ZipFile(root/'source_bundle.zip') as archive:
24:         if len(archive.namelist())!=len(names) or set(archive.namelist())!=set(names):
````

## s056

来源：`learning_pipeline/server/delivery_contract.py:38`

````text
38: def verify(root,pending=()):
39:     root=Path(root).resolve();manifest=json.loads((root/'manifest.json').read_text(encoding='utf-8'))
40:     pending=set(pending)
41:     if not pending<=HOST_FILES:raise RuntimeError('Only declared environment helpers may be adjusted')
42:     with zipfile.ZipFile(root/'source_bundle.zip') as z:
43:         names=[m['path'] for m in manifest['members']]
44:         if len(names)!=len(set(names)) or len(z.namelist())!=len(names) or set(z.namelist())!=set(names):raise RuntimeError('Member set differs')
45:         if z.testzip():raise RuntimeError('CRC failed')
46:         latest,records=adjustments(root,z,manifest['version'])
````

## s057

来源：`learning_pipeline/evaluation/runtime_ipc.py:1`

````text
1: """Allocate a short private vLLM RPC directory without changing task TMPDIR."""
2: import os
3: from pathlib import Path
4: import socket
5: import tempfile
6: import uuid
7: 
8: 
9: def allocate(resource_dir, gpu_index, *, keep=False):
````

## s058

来源：`README.md:67`

````text
67: r01两臂均在vLLM IPC路径120B超过107B时启动失败，真实0题、清理闭合，原失败证据保留。根因是本地临时路径设计遗漏，后续summary缺失是次生错误。新r02只设置官方支持的短`VLLM_RPC_BASE_PATH`并实测socket；提示/LoRA/预算/SDK裁判保持。
68: 
69: 只上传[5件小差量目录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/dist/harness_k_a800/trajectory_cycle01_joint_t02_t10_pair_r02_update)（80426B），把[完整zcode指令](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/dist/harness_k_a800/trajectory_cycle01_joint_t02_t10_pair_r02_update/ZCODE.txt)转交。服务器保留r01原件，从旧封存包复用大输入到新r02目录，继续GPU2 T0.2/GPU3 T1.0各129一次并结束后保留本人预留；不清旧排他文件、不重传207MB、不重跑已执行题。[原件分析与验收](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/reports/trajectory_cycle01_pair … [长行省略]
70: 
71: ## 2026-11-14：D082 首轮LoRA训练已接纳并提交，准备双温度129评测
72: 
73: 四卡FSDP2 r05实际完成498更新/两epoch，最长19356-token两次累积通过；820训练张量与840服务张量逐值核验、真实vLLM挂载检查完成。单干净GPT-6.1 Sol/max已接纳训练与原六成员ZIP，首次失败load和总失败回执保留，按最终续跑证据作新增接纳。训练约10.30h；加载数值变化不等同解题提升。
74: 
75: 用户本轮授权实际Kaggle提交：[SELF-C01-D03-T02-r01包](dist/self_c01_d03_t02_kaggle_r01/submission.zip)194789594B，提交号<OPERATION_ID>，目前PENDING，尚无新成绩。原0.18提交及所有基线保留。[回传审查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/reports/trajectory_cycle01_fsdp4_r05_return_review_20261114)记录原件、独审、完成状态解释和提交回执。
````

## s059

来源：`learning_pipeline/evaluation/panel_driver.py:113`

````text
113: def cleanup_closed(arm_id):
114:     if any(CONTROL.rglob(arm_id+'*.unclosed.json')):return False
115:     p=paths(arm_id);report=p['report']
116:     if (report/'batch_result.json').is_file():return read(report/'batch_result.json').get('safe_to_pack') is True
117:     # A preflight failure has no model path; the wrapper process must have exited.
118:     return not (report/'batch_started.json').exists() and not p['output'].exists() and not p['setup'].exists()
119: 
120: 
121: def worker(candidate):
````

## s060

来源：`reports/trajectory_cycle01_pair_r02_review_20261115/OVERVIEW.md:38`

````text
38: ## 回传为什么大，已经怎样缩小
39: 
40: 原tar.gz **1706923496B**，23836存储成员；恢复5个精确引用后逻辑文件23841（含manifest）。存储层展开体积中，8个ZIP约1.414GB加一个权重247.36MB，占约76%；多数ZIP里面又重复装着同一个adapter。仅做整个文件去重不够。
41: 
42: 本地用全部实际原件制作analysis验证包 **154622101B（154.6MB）**，减少**90.94%**。所有**21093 HTTP文件/7031组**、**1030 ADK/observations文件**、官方trace、patch、验证、GPU与清理记录保持原字节。全内容核对为23834普通/引用逻辑文件及364个ZIP成员通过；少传的是与已接纳冻结输入全字节相等的权重/ZIP重复内容，未知或不同字节照常保留。
43: 
44: 需要保留本地冻结`dist/self_c01_d03_t02_kaggle_r01/submission.zip`才能完整恢复引用。嵌套ZIP只保证全部成员字节，不保留原压缩容器字节；原完整tar、旧发行和服务器原件未改。首次新adapter仍必须传真实权重，未来完整自含需求有`--profile full`。当前源码已接入未来pair打包与统一lean-return/restore-return入口；本轮不重新上传或重新跑服务器。
45: 
46: 验证：[真实全内容直比](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/lean_return_verification.json)、[独立瘦身审查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/lean_return_review.md)。本地13项初检及3组可复用回归覆盖不同权重、Unicode、引用尺寸/key/CRC、原路径恢复和拒绝覆盖。初次unittest被Windows Temp目录ACL拦住，尚未进入内容测试；改用工作区新建隔离测试目录后全过，未改生产验收规则。
````

## s061

来源：`reports/checkpoint_panel_c03_20261116/overview.md:27`

````text
27: ## 交付与回传
28: 
29: 五件压缩上传约12.5MB；source_bundle.zip 12,509,850B /65成员，字节与展开/当前源直接一致。README另按独审澄清CPU序列化fixture存在。只上传该新目录，避免再次同步训练目录。实际大小清单与最后直接绑定见delivery_receipt.json。
30: 
31: analysis回传8臂所有HTTP/ADK/patch/verification/官方trace/日志/配置/SDK路由/耗时/显存/清理/预留；中间权重、八臂精确输入ZIP保留服务器，外部引用明确NOT_RETURNED。本地可完整读行为证据，但不能声称已独立直比中间权重；选定后再取该候选准确温度的输入。--full可自含。原C03训练回传独立保留。
32: 
33: 无新hash、模型下载、依赖安装、服务器操作、教师、训练、129或Kaggle提交。旧训练发行和历史数据/adapter不变。
````

## s062

来源：`learning_pipeline/specs/FRESH_SFT_C03.md:14`

````text
14: | 任务口径 |原SELF-v17-r04六工具/28调用/480秒/80轮，240秒命令/300秒裁判、8192总输出/4096思考保持 |
15: | 服务器范围 |训练沿用0–3，短加载GPU2，结束恢复本人GPU2/3预留；GPU0他人条目不碰，4–7不操作 |
16: 
17: 发布身份：`trajectory_cycle03_fresh_fsdp4_r01`。数据身份：`learning_runs/TC-r04-001/datasets/D05_fresh`，复用该轮无adapter的6842B Agent来源；新数据拥有独立来源/选择/派生/资格账本，原round catalog和原D03/D04不可变。最终N=990、160个任务ID，U=124/W=7，保存31/62/93/124，末批6条按2/2/1/1分配（按真实长度决定rank）。
18: 
19: ## 本次实际冻结结果
20: 
21: | 项目 | 实际值 |
22: |---|---:|
````

## s063

来源：`archive/harness_k_a800/trajectory_cycle02_joint_t02_t10_pair_r01/project/calibration/coverage.json:1`

````text
1: {
2:   "all_task_ids": [
3:     "fastapi_11194",
4:     "fastapi_11355",
5:     "fastapi_12942",
6:     "fastapi_13207",
7:     "fastapi_13537",
8:     "fastapi_13713",
9:     "fastapi_13786",
````

## s064

来源：`runtime/baseline.py:351`

````text
351:                 summary = {
352:                     "instance_id": task.instance_id, "resolved": result.resolved,
353:                     "test_exit_code": result.test_exit_code, "patch_chars": len(result.agent_patch),
354:                     "tool_calls": result.tool_calls, "llm_calls": result.total_llm_calls,
355:                     "duration_seconds": result.duration_seconds, "error": result.error_message,
356:                     "evaluation_wall_seconds": time.monotonic() - task_started_at,
357:                     # The official context freezes this clock when its Agent
358:                     # session stops; TaskResult duration also includes other work.
359:                     "agent_duration_seconds": context.agent_elapsed_seconds,
````

## s065

来源：`learning_pipeline/contracts/teacher_fragment_review.md:11`

````text
11: - 只接收 `job_result.status=FRAGMENT_CAPTURED`、`collection_mode=fragment`、`overall_verification=NOT_RUN`。`final_stage` 指最后真实下一请求边界，不是整题终态。该链没有新 TaskResult、独立判分或虚构 submit/final。
12: - `teacher_000` 到末项全部逐项处置，包括中途实际 summary；编号完整且唯一。每个实际提案须有 ACCEPTED 的 `response.json`、原 `proposal.txt`、`response.raw.json`、实际 `teacher_input.txt`、无 native 工具的 CLI 事件，以及下一 stage 的实际请求/状态比较和 ADK 调用及后续回执。`target_binding` 的执行计划不能代替 `execution_binding.status=EXECUTED`。
13: - 所有 stage（包括前缀 stage_000 和末 stage）必须真实 BOUNDARY_CAPTURED、cleanup 完成、`protected_mutation_observed=false`、cleanup 前保护字节相等、无 audit_error，并保留下一请求与 task_state.zip/json。首次原学生工作区未观测仍记 RECONSTRUCTED，不称严格同现场。
14: - 源码写入、验证或捕获结论、正 reasoning、保护/历史访问边界，须由独立审查者或执行协调员实际核实。不得因工具 ok 就认可未知代码。保护污染、历史/私有证据访问、未执行或工具拒绝不能正训。
15: - 非零检查仅在它确实是有效失败诊断时可接纳：lane 必须 VALID_DIAGNOSTIC，另写 `receipt_assessment=EXPECTED_DIAGNOSTIC_FAILURE`，并解释真实失败揭示的缺口。畸形脚本、错误断言或缺参拒绝不属于此例外。
16: 
17: ## 逐目标资格
18: 
19: 每个 target 的三个 approved 字段都是真 boolean。未选、错误、未知、summary 的三个字段均 false；不要为补齐行数降低标准。短片段动作仅能标 LOCAL_ACTION_VERIFIED 或 VALID_DIAGNOSTIC。永远不标 FULL_REPAIR_VERIFIED，`final_approved` 永远 false。
````

## s066

来源：`learning_pipeline/collection/runner.py:282`

````text
282:             if p['decision_summary']:msg['reasoning_content']=p['decision_summary']
283:             if p['kind']=='tool_call':msg['tool_calls']=[{'id':f'teacher-{task_id}-{cut}-{attempt}-{idx:03d}','type':'function',
284:                 'function':{'name':p['name'],'arguments':json.dumps(args,ensure_ascii=False)}}]
285:             write_new(teacher/'response.raw.json',{'model':'controlled-gpt-6.1-sol/max','choices':[{'index':0,'message':msg,
286:                  'finish_reason':'tool_calls' if p['kind']=='tool_call' else 'stop'}],'usage':None,'teacher_usage_separate':response.get('usage')})
287:             write_new(teacher/'target_binding.json',{'input_request':str(stage/'next_request.json'),'actual_proposal':str(teacher/'proposal.txt'),
288:                 'author':'gpt-6.1-sol/max','executed_in_next_stage':idx+1,'training_eligible':False,
289:                 'reasoning_source':'TEACHER_PUBLIC_DECISION_SUMMARY','private_teacher_thinking_collected':False})
290:             responses.append({'mode':req['mode'],'response_path':linux(teacher/'response.raw.json'),'source':'SOL_TEACHER',
````

## s067

来源：`learning_pipeline/SPEC.md:276`

````text
276: ## 8. thinking与公开决策说明
277: 
278: 本轮用户要求与A–E同时推进推理引导。**129题每个实际可见reasoning回合均审查**，但不预设每段都错或每段都值得正训。主审与另一分片交审均确认该整段仅依据当时输入、无错误事实/未来泄漏/无效重复，并与同回合实际合格动作相符，才可监督原文；混有错误或未知内容则整段mask，不事后删改成“原始正确thinking”。摘要回合只审其状态影响，不作为动作reasoning正例。没有可见reasoning就记ABSENT，不补造。
279: 
280: Sol/Codex私有思维链不请求、不记录。真正新教师在实际动作前另行输出**公开简短决策说明**：已观察证据与下一动作/检查的理由，一般几句。它是教师专门撰写的训练内容，来源固定记`TEACHER_PUBLIC_DECISION_SUMMARY`，不是其内部思维链。输出由干净CLI进程产生，后续真实官方工具执行，资格再独审；分析者看完结果写的rewrite_suggestion永远只是metadata，不进loss。
281: 
282: 原生模板将工具回合的reasoning_content放入thought channel。新导出器仅监督本目标中被接受的thought区段及工具/终答区段，旧学生错误、已有老师前缀、工具观察全部-100。分别导出`labels_action`、`labels_reasoning`和合并labels；同一行的action-only对照保持完全相同的input_ids，仅改mask，不能删掉因果历史制造更容易输入。
283: 
284: 新数据在原生span/token前缀与独立mask审查通过前不冻结。训练建议先比较action-only与联合监督；联合版按组件分别平均再加权，避免长thought淹没短动作。具体混合权重在训练配方明列并可审；当前无GPU训练，不称其必然提分。原学生prompt/推理预算不改变。
````

## s068

来源：`learning_pipeline/native.py:43`

````text
43:     def example(self,sample_id,request,target,selected_call_ids,metadata,reasoning=False,final_text=False):
44:         target=copy.deepcopy(target)
45:         prompt=self.render(request['messages'],request.get('tools'),True)
46:         full=self.render(request['messages']+[target],request.get('tools'),False)
47:         if not full.startswith(prompt):raise ValueError('Native text prefix does not match')
48:         pe=self.tokenizer.encode(prompt,add_special_tokens=False)
49:         fe=self.tokenizer.encode(full,add_special_tokens=False)
50:         if fe.ids[:len(pe.ids)]!=pe.ids:raise ValueError('Strict native token prefix mismatch; no truncation/fallback')
51:         completion=full[len(prompt):]
````

## s069

来源：`reports/fresh_lora_implementation_20261116/overview.md:60`

````text
60: 全行native审计和一个干净Sol/max独审均通过：**990行/12,199,343位置**的源对象、允许的派生、完整正标签、action/reason分解、权重及split全核；无历史/回执监督。六主件快照61,075,049B逐成员核同。独立审查使用自身边界和offset重算，不调用本次builder的complete_example自证。[冻结审查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/review.json) · [发行审查](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/release_review.json)。
61: 
62: 必要CPU检查包括trainer的CE/梯度/尾批/快照、builder多回合源重建及负例、实际服务器合同和旧入口回归。独审自行执行20项及A差量1项，复用实施侧检查结果而不反复扩审。实际本地Torch2.14CPU与目标2.10CUDA的边界保留；本地PASS不等于31B训练/加载已经运行。
63: 
64: ## 旧结果与下一轮边界
65: 
66: 用户截图显示C01-D03-T10-r01 **0.15**，同LoRA T02 **0.17**，均Succeeded；本轮未重新API核提交ID，原截图和[观测记录](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/prior_c01_scores_20261116.json)已保存。它们没有达到0.20/0.22目标；T1本地52/129没有形成线上温度优势。不能由两个提交单独诊断LR、rank或灾难遗忘，也不能因为新包工程验收通过就承诺新分数。
67: 
68: 本轮实际状态为 **DATA_FROZEN_HANDOFF_READY**。服务器31B训练、加载、候选行为选择、129、Kaggle均未新增运行；GPU与预留由zcode按已授权交接实际处理。下一步是上传当前七件并转交完整指令，收到实际训练与checkpoint证据后继续选择和评测。
````

## s070

来源：`learning_pipeline/fresh.py:159`

````text
159: def _a_source(native,record,index):
160:     """Precisely match one author-converted example to its message and prefix."""
161:     assistants=[i for i,m in enumerate(record['messages']) if m.get('role')=='assistant']
162:     if len(assistants)!=len(record['sft_examples']):raise ValueError('A assistant/example alignment differs')
163:     message_index=assistants[index];history,removed=_served_a_messages(record['messages'][:message_index])
164:     target=copy.deepcopy(record['messages'][message_index]);example=record['sft_examples'][index]
165:     canonical_prefix=native.render(history,record['tools'],True)
166:     # The published A corpus was built with a source generation boundary that
167:     # forces the thought-channel opener after a tool response.  Our deployed
````

## s071

来源：`learning_pipeline/specs/FRESH_SFT_C03.md:21`

````text
21: | 项目 | 实际值 |
22: |---|---:|
23: | 本地D04重选 |720回合/113任务ID；原2090中1370排除、16题不凑只读样本 |
24: | 公开Native |233回合/41任务ID |
25: | 公开A |37回合/6任务ID；source-converted，不冒称本SDK重执行 |
26: | 合计 |990回合/160任务ID；新增外部47任务来自45个训练repo |
27: | 主要阶段 |定位313、编辑271、验证195、恢复收尾211 |
28: | 完整输入位置 |12,199,343 |
29: | 正目标token |199,458＝156,980行动/边界＋42,478合格公开说明 |
````

## s072

来源：`reports/fresh_lora_implementation_20261116/overview.md:15`

````text
15: | 来源 | 目标回合 | 任务ID | 实际目标系数质量 |
16: |---|---:|---:|---:|
17: | 本地Gemma合格行为（含选中旧来源） |245|65|27.9219% |
18: | 本地Sol纠偏（含选中旧来源） |475|76|42.7031% |
19: | 公开Native |233|41|25.6250% |
20: | 公开A转换来源 |37|6|3.7500% |
21: | 合计 |**990**|**160**|**100%** |
22: 
23: 本地两行任务有交集，不能将65+76相加当141道不同题。本地总体113任务ID；外部47任务ID/45训练repo。本地从D04的2090行逐题重选720，排除1370行；16题只有准备性读取/诊断，不为凑129给整题训练权重。旧资格复用，未重审全部旧HTTP。外部train分片读完47题/85attempt/1532可见回合，205段保留的非空说明全文批准。没有本轮新教师生成。
````

## s073

来源：`learning_pipeline/specs/FRESH_SFT_C03.md:28`

````text
28: | 完整输入位置 |12,199,343 |
29: | 正目标token |199,458＝156,980行动/边界＋42,478合格公开说明 |
30: | 完整长度 |最长26,910/32768；最大当前completion2208/8192，无截断 |
31: | 说明及收尾 |721行有原合格说明，7行事实final；0新教师调用 |
32: | 均权 |全行平均loss倍率1，范围0.1546875–3.09375 |
33: | 实际目标系数 |行动72.6095%、说明27.3905%；这是优化目标系数，不是测得的梯度份额 |
34: | 外部隔离 |dev/final各8repo9issue，未入训练；环境尚未重建 |
35: 
36: 独立Sol/max逐源对象核变换、全文位置/两component mask/倍率/来源资格及split：990行、12,199,343位置全过，开放阻断0。六主件快照61,075,049B逐成员直接核同，已由统一freeze接纳。21个既有真实serving渲染fixture继续逐字/逐token相等。语义审查复用已有本地资格，外部两个分片读完85条轨迹1532可见回合；不冒称所有历史reasoning都全文审过。
````

## s074

来源：`learning_runs/TC-r04-001/datasets/D05_fresh/summary.json:7`

````text
7:   "rows": 990,
8:   "tasks_with_targets": 160,
9:   "canonical_tasks_with_targets": 160,
10:   "token_counts": {
11:     "input": 12199343,
12:     "action": 156980,
13:     "reasoning": 42478,
14:     "reasoning_rows": 721,
15:     "final_rows": 7
````

## s075

来源：`reports/fresh_lora_implementation_20261116/overview.md:25`

````text
25: 阶段为定位313、编辑271、验证195、恢复收尾211。`run_command`按真实内容分类，不能全部算探索。完整输入12,199,343位置，正token **199,458**（action/合法边界156,980＋合格公开说明42,478）；721行说明、7行事实final。最长26,910，无截断；最大目标2208，未超过8192输出预算。新视图是完整合格completion CE，不把未合格当前说明留在输入中当答案提示；原件和显式移除记录保留。
26: 
27: 每题/阶段总质量均衡，均匀遍历使用`m=N/(T*K_i*n_i_s)`，平均1，范围0.1546875–3.09375。由真实行长度和倍率算得说明目标系数27.3905%、行动72.6095%，不是简单token比例，也不是测得梯度贡献。详细[计数与系数](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/training_dataset_statistics.json)。
28: 
29: 数据有价值的筛选实例：
30: 
31: - 本地保留12942的递归Annotated修正及真实端点/兼容检查，11194的字段类型修正链；普通查询、重复成功声明和无实质推进的状态/提交不单独准入。
32: - 外部发现有edit解释错误地声称旧行为完全保留、正确源码调用却夹着错误推理的情况：动作可有资格，错误当前说明在新派生视图移除，不伪造新思考。
33: - speedcopy的abspath改动可能改变返回值接口，没有因源resolved就正训整edit；CrossGL的exit0来自catch但解析实际失败，不标成功验证。
````

## s076

来源：`learning_pipeline/specs/FRESH_SFT_C03.md:44`

````text
44: - 本地D04复用已通过的原生mask、动作语义及公开说明资格；重选阶段价值，避免再次审阅全部历史HTTP。每题保留必要定位、源码编辑、命中真实行为的验证、恢复/事实收尾；既有源身份不是质量分数，不能只选教师或只选成功链。
45: - 公开候选只Native/A。训练split47issue/85轨迹由两个干净Sol/max curator分24/23题阅读其全部动作/回执链；正标签reasoning必须全文已读并接受，历史未读reasoning继续masked。dev/final各9issue/8repo不用于筛选目标；B/B2不扩入首轮。
46: - 每题选主要有效路径，可保留互补真实状态。无信息重复、错误/未知源码、工具格式错、假成功不进正标签。不得把源resolved当本SDK重执行通过；保留SOURCE_NATIVE_CAPTURE_SUPPORTED与SOURCE_CONVERTED_SUPPORTED。
47: - 新派生视图显式替换为当前system和六工具声明，原user题面、过去动作和工具回执保持；当前目标原动作不改。只监督完整合格completion（原已核公开说明＋动作/合法结束），不训练工具回执、后验理由或私人thinking。
48: - 未合格当前说明从新衍生输出移除，保留原件和变换记录；不把其masked后继续作为答案提示。不截断长说明；80token只是研究短说明偏好，不是准入硬门。多工具目标只有全部目标动作获资格才可完整监督，否则隔离原因。
49: - 现有joint/action历史模式原样，新`completion`模式对完整正token逐行平均CE；分别记录action/说明诊断。每题阶段均衡采用`loss_weight=N/(T*K_i*n_i_s)`，平均倍率1。均匀行遍历，不再按同一权重重采样。
50: - 重复只在同源/等价原状态有直接证据时去重；相似状态只标冗余筛选，不假称精确去重。长于32768的输入隔离而不截断。来源、schema异常、隐私替换按实际影响判断；本地采集guard不冒充官方禁止规则。
51: 
52: ## 入口与验证
````

## s077

来源：`reports/fresh_lora_implementation_20261116/overview.md:36`

````text
36: 各分片真实选择/排除和原件锚：[本地](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/local_selection/report.md)、[外部A分片](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/external_a/report.md)、[外部B分片](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/external_b/report.md)。Native原生捕获支持与A转换来源支持明确分开，均未宣称本轮在本SDK重新执行。dev/final各8repo9issue没有进入训练；它们不是已跑通的外部环境，129本地题早已暴露。
37: 
38: ## 实际配方及已实现能力
39: 
40: | 参数 | 本轮固定值 |
41: |---|---|
42: | 底座/初始化 |官方QAT恢复冻结BF16；PEFT随机A/零B、新AdamW，无父adapter |
43: | LoRA |标准r16/alpha16/dropout.05，全60层410投影，122,429,440参数 |
44: | batch与长度 |4卡micro1×accum2=global8；1pass、**124更新**，尾6不补样 |
````

## s078

来源：`learning_pipeline/training/SOURCES.md:13`

````text
13: ## 第一方版本依据
14: 
15: - [Transformers 5.13.1 Gemma4接口](https://huggingface.co/docs/transformers/v5.13.1/en/model_doc/gemma4)：核对实际 `Gemma4ForConditionalGeneration`、原生工具模板与文本配置；不把普通31B或新版接口替代现有QAT父模型。
16: - [Transformers 5.13.1 compressed-tensors量化器](https://github.com/huggingface/transformers/blob/v5.13.1/src/transformers/quantizers/quantizer_compressed_tensors.py)：`run_compressed=False`在加载后走decompress_model。归档同版本wheel源也已检查，沿用此API而不改成后续版本的参数。
17: - [PEFT 0.21.2 LoRAConfig源码](https://github.com/huggingface/peft/blob/v0.21.2/src/peft/tuners/lora/config.py)：核对target_modules、rank/alpha/dropout与bias参数。该版本官方文档URL本次不能访问，因此版本特定判断以实际wheel/源码及已有真实CPU序列化检查为准；不将重定向到0.21.0的文档冒充0.21.2。
18: - [vLLM 0.19.1 LoRA装载资料](https://docs.vllm.ai/en/v0.19.1/features/lora/)：支持独立LoRA请求。实际比赛build仍按 `official_lora_sources.zip` 的14份源文件直接比较；不能仅靠同为0.19.1就宣称一致。
19: - [Google Gemma的Transformers入口](https://ai.google.dev/gemma/docs/core/huggingface_inference)：仅作为官方模型使用来源，不证明本A800训练配方或比赛四卡部署已通过。
20: 
21: ## D073训练交接修复
````

## s079

来源：`learning_pipeline/training/train_core.py:266`

````text
266: def preserve_explicit_lora_scope(model,base,targets):
267:     """Keep saved PEFT scope exact after its optional suffix minimization."""
268:     wanted=set(targets)
269:     actual={name for name,module in base.named_modules() if 'default' in getattr(module,'lora_A',{})}
270:     if actual!=wanted:raise ValueError('Actual LoRA injection differs from the explicit text projection set')
271:     # Serialization only: injection is already complete. Avoid broad suffixes
272:     # expanding into other towers when an adapter is loaded on a full model.
273:     model.peft_config['default'].target_modules=wanted
274:     return sorted(actual)
````

## s080

来源：`learning_pipeline/SPEC.md:148`

````text
148: ## D081：QAT辅助状态与FSDP2兼容（2026-11-14）
149: 
150: r04原包188352B/110文件（109payload+summary）完整归档。四rank同在PyTorch2.10 nn.Parameter(DTensor)默认requires_grad=True时报非浮点错误，无31B longest前向/反向或优化更新。普通小Gemma在真实2.10CUDA全过，不能据此宣称31B显存稳定。原日志不含失败参数名；RoPE/zero_point不是已证根因。
151: 
152: 固定compressed-tensors 0.15.0.1源码及真实小Linear压缩/解压证实：pack-quantized解压删除weight_packed，但保留weight_shape int64 Parameter和浮点weight_scale。官方2.10包装参数的先后顺序与错误栈吻合；没有现场清单前，具体31B触发参数仍标强候选。新版在31B分片前落盘全部参数名称/shape/dtype/bytes/grad供直接定位。
153: 
154: 修复只按属性分类：decoder冻结非浮点Parameter在各layer与root均忽略分片，保留原对象、dtype和同步后值；不转float、不删除、不转buffer、不动依赖库。冻结浮点weight/scale仍分片，820 FP32 LoRA仍复制显式归约。首次广播前全rank名称/shape/dtype顺序一致；记账分开decoder浮点可分片字节、非浮点复制字节与实际local逻辑字节，容量仍看CUDA实测而非参数理论量。
155: 
156: 回归升级现有小Gemma检查，不追加长CPU门禁：真实CT symmetric int4/group32 compress→decompress，出现weight_shape/weight_scale后再PEFT和四rankFSDP；含旧默认Parameter(int)负控、8→6→8、最长两累积、dropout重算、冻权/元数据逐值、PEFT存取。该测试是module lifecycle，不冒称31B原checkpoint加载。通过后沿现有长样本/首步→两epoch训练链直接执行。D03/全部输入及mask/19356/BF16/超参/Agent/GPU0–3与保护策略不变；新execution版本r02，新发行trajector … [长行省略]
````

## s081

来源：`learning_pipeline/training/fsdp_runtime.py:1`

````text
1: """Shard frozen decoder weights; explicitly average replicated LoRA gradients.
2: 
3: The shared embedding/head stay replicated: selected-token CE invokes the head a
4: different number of times on different ranks. Never put that head in an FSDP
5: collective. A parameterless objective root coordinates the decoder FSDP states.
6: """
7: import torch
8: import torch.distributed as dist
9: 
````

## s082

来源：`learning_pipeline/training/fsdp_runtime.py:49`

````text
49:     # All ordinary tensors left outside managed decoder groups stay replicated.
50:     ignored={p for p in objective.parameters() if not isinstance(p,DTensor)}
51:     fully_shard(objective,mesh=mesh,reshard_after_forward=True,ignored_params=ignored)
52:     del ignored,ignored_layer_parameters
53:     if objective._get_fsdp_state()._fsdp_param_group is not None:
54:         raise ValueError('Objective root unexpectedly owns a parameter group')
55:     for layer in layers:
56:         for p in layer.parameters():
57:             if not p.requires_grad and p.is_floating_point() and not isinstance(p,DTensor):raise ValueError('Floating decoder weight left unsharded')
````

## s083

来源：`learning_pipeline/training/SOURCES.md:36`

````text
36: ## D077 MLP分块重算与四rank
37: 
38: 实际失败点来自固定[Transformers 5.13.1 Gemma4TextMLP](https://github.com/huggingface/transformers/blob/v5.13.1/src/transformers/models/gemma4/modeling_gemma4.py)及[PEFT 0.21.2 Linear](https://github.com/huggingface/peft/blob/v0.21.2/src/peft/tuners/lora/layer.py)。MLP按位置独立，直接复用原bound forward，以512-token块运行，不改公式、投影或参数键名；KV共享区的加宽MLP … [长行省略]
39: 
40: 依据[PyTorch 2.10 checkpoint](https://docs.pytorch.org/docs/2.10/checkpoint.html)，采用non-reentrant及preserve_rng_state=True；原decoder外层checkpoint仍在。固定分块的dropout重算必须等价，分块前后随机数分配不保证逐位相同。实际固定TF/PEFT微型数值检查包含普通/加宽MLP、非整除尾块、嵌套重算、LoRA输入/参数梯度与RNG；不是31B CUDA峰值证明。
41: 
42: 四rank每rank累积2，保持global8/498更新；尾6分2/2/1/1，各rank以最后一个真实micro同步，梯度平均后仍sum/8。真实四Gloo进程覆盖full8→tail6→full8及MLP嵌套重算。最长完整序列预检同时保留同形同dtype的两份AdamW状态容量，仅预留容量而非提前optimizer.step；恢复RNG/清梯度后，首个原洗牌批真正更新并验证AdamW状态，记入498步。
43: 
44: 旧双卡r01两次目标机OOM/零更新证据保留。本轮四A800实际长序列前反向、吞吐、adapter加载与成绩全部待服务器回传，不预报速度或提分。
````

## s084

来源：`learning_pipeline/training/SOURCES.md:9`

````text
9: - [Scott Weeden 的公开训练kit](https://www.kaggle.com/datasets/scottweeden/gemma-4-developer-agent-tool-lora-kit)：本地归档为 `vendor/lora_research_20261112/scott_kit/4/scripts/train.py`（43326字节）及 `0/recipe/gemma4_31b_peft_recipe.md`（6607字节）；原始来源和许可记录见该目录外的 `SOURCE_NOTES.txt`。实际代码提供了按query分块的checkpoint attention、仅目标位置的chunked CE、 … [长行省略]
10: - 上述作者实际脚本还使用NF4，并在注意力注册失败时将长度降至6144、过滤长行；这些行为不进入本框架。我们保留完整32768上限，任何超长行或注册失败都显式失败。原PEFT adapter保持不变，按实际competition wheel另做K→V服务导出，不套作者宽后缀配置。
11: - 作者榜上最佳成绩没有绑定到这一份adapter/脚本，不作为本包效果证据。其他已归档教程存在Gemma2/随机fallback、模拟训练或全token标签问题，未作为运行实现采用。
12: 
13: ## 第一方版本依据
14: 
15: - [Transformers 5.13.1 Gemma4接口](https://huggingface.co/docs/transformers/v5.13.1/en/model_doc/gemma4)：核对实际 `Gemma4ForConditionalGeneration`、原生工具模板与文本配置；不把普通31B或新版接口替代现有QAT父模型。
16: - [Transformers 5.13.1 compressed-tensors量化器](https://github.com/huggingface/transformers/blob/v5.13.1/src/transformers/quantizers/quantizer_compressed_tensors.py)：`run_compressed=False`在加载后走decompress_model。归档同版本wheel源也已检查，沿用此API而不改成后续版本的参数。
17: - [PEFT 0.21.2 LoRAConfig源码](https://github.com/huggingface/peft/blob/v0.21.2/src/peft/tuners/lora/config.py)：核对target_modules、rank/alpha/dropout与bias参数。该版本官方文档URL本次不能访问，因此版本特定判断以实际wheel/源码及已有真实CPU序列化检查为准；不将重定向到0.21.0的文档冒充0.21.2。
````

## s085

来源：`reports/trajectory_cycle02_fsdp4_r01_return_review_20261116/overview.md:13`

````text
13: | 导出/服务 | 820→840 BF16逐值正确，195042419B六成员单adapter ZIP，真实vLLM短加载通过 | 可作为评测候选；短探针不等于解题提分 |
14: 
15: ## 需要纠正的loss口径
16: 
17: 全524步的全局损失都不是0。末步524只有2条真实样本，全局均值**0.00413464645**；rank0 rich3905的18目标token为0.00000260272，rank1 fastapi5077的48目标token为0.00826669019，rank2/3没有真实行、只有零loss通信槽。将单卡近零值、空槽或少量简单目标写成“训练loss已归零”会夸大结论。全局日志聚合代码正常，不需要为此重跑训练。
18: 
19: `mean_supervision_loss`是行内行动/说明联合损失的真实样本均值；反向传播另有task均权。上述每轮均值描述同一批数据在持续更新过程中的日志，不是固定checkpoint重放或独立验证。C01和C02数据构成不同，不能用两轮训练的绝对loss直接判模型谁更强。
20: 
21: ## 梯度不是持续发散，但也不能说毫无波动
````

## s086

来源：`reports/trajectory_cycle02_fsdp4_r01_return_review_20261116/overview.md:29`

````text
29: ## 参数变化与能力变化分开判断
30: 
31: 本轮122429440个LoRA参数中122429400个值改变；原生adapter整体ΔL2/L2父=0.1259964，最大单值变化0.00457803。这是参数空间的变化描述，不是“能力提升12.6%”。继承父权重、首步真实更新、完整样本消费与权重变化共同支持续训不是空跑。
32: 
33: 本轮47549010输入token，比C01的40296066多约18%，最长26979也高于19356；吞吐971.6对1087.2不是同工作量性能基准。当前batch/序列保持即可，不因为allocated52.7就推断可以直接扩大批次。
34: 
35: ## 下一步判断标准
36: 
37: 建议保持当前prompt、六工具与预算，进入无教师129题开发对照，分别比较C02 T1.0与C01的52、C02 T0.2与C01的48。除总resolved外，重点看已定位后及时修改、真实入口/兼容范围、有效触发检查、空patch及无效继续生成，并确认传参改善没有回退。129题已经训练暴露，不能把该分数称为独立泛化，Kaggle结果另行观察。
````

## s087

来源：`reports/trajectory_cycle02_fsdp4_r01_return_review_20261116/overview.md:15`

````text
15: ## 需要纠正的loss口径
16: 
17: 全524步的全局损失都不是0。末步524只有2条真实样本，全局均值**0.00413464645**；rank0 rich3905的18目标token为0.00000260272，rank1 fastapi5077的48目标token为0.00826669019，rank2/3没有真实行、只有零loss通信槽。将单卡近零值、空槽或少量简单目标写成“训练loss已归零”会夸大结论。全局日志聚合代码正常，不需要为此重跑训练。
18: 
19: `mean_supervision_loss`是行内行动/说明联合损失的真实样本均值；反向传播另有task均权。上述每轮均值描述同一批数据在持续更新过程中的日志，不是固定checkpoint重放或独立验证。C01和C02数据构成不同，不能用两轮训练的绝对loss直接判模型谁更强。
20: 
21: ## 梯度不是持续发散，但也不能说毫无波动
22: 
23: 实际515/524步的裁剪前范数超过1；step182为7984.323，随后第183步27.957、第184步3.926，后续日志和最终权重均有限。执行代码在AdamW前以max_norm=1.0裁剪，日志记录的是返回的裁剪前范数。该语义可对照[PyTorch2.10官方说明](https://docs.pytorch.org/docs/2.10/generated/torch.nn.utils.clip_grad_norm_.html)。不能把7984称为裁剪后的更新幅度，也不能由“没有NaN”推断它一定有益。
````

## s088

来源：`reports/trajectory_cycle02_fsdp4_r01_return_review_20261116/overview.md:21`

````text
21: ## 梯度不是持续发散，但也不能说毫无波动
22: 
23: 实际515/524步的裁剪前范数超过1；step182为7984.323，随后第183步27.957、第184步3.926，后续日志和最终权重均有限。执行代码在AdamW前以max_norm=1.0裁剪，日志记录的是返回的裁剪前范数。该语义可对照[PyTorch2.10官方说明](https://docs.pytorch.org/docs/2.10/generated/torch.nn.utils.clip_grad_norm_.html)。不能把7984称为裁剪后的更新幅度，也不能由“没有NaN”推断它一定有益。
24: 
25: 作为背景，C01已有470/498步超过1、最大7792.382；本轮裁剪更频繁，但这个量级的孤立峰并非首次出现。尚无逐样本梯度或step182前后参数/优化器快照，不将尖峰归因于某一道题、不宣称精确裁剪后实测范数。当前证据支持保留为观察项，先做行为评测，不据此直接改学习率或追加epoch。
26: 
27: ![C02训练损失与裁剪前梯度](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/training_curve.png)
28: 
29: ## 参数变化与能力变化分开判断
````

## s089

来源：`learning_pipeline/specs/FRESH_SFT_C03.md:17`

````text
17: 发布身份：`trajectory_cycle03_fresh_fsdp4_r01`。数据身份：`learning_runs/TC-r04-001/datasets/D05_fresh`，复用该轮无adapter的6842B Agent来源；新数据拥有独立来源/选择/派生/资格账本，原round catalog和原D03/D04不可变。最终N=990、160个任务ID，U=124/W=7，保存31/62/93/124，末批6条按2/2/1/1分配（按真实长度决定rank）。
18: 
19: ## 本次实际冻结结果
20: 
21: | 项目 | 实际值 |
22: |---|---:|
23: | 本地D04重选 |720回合/113任务ID；原2090中1370排除、16题不凑只读样本 |
24: | 公开Native |233回合/41任务ID |
25: | 公开A |37回合/6任务ID；source-converted，不冒称本SDK重执行 |
````

## s090

来源：`reports/fresh_lora_implementation_20261116/training/implementation.md:13`

````text
13: - final 仅保留一份 AdamW，另有各 rank Python/Torch CPU/本地 CUDA RNG（numpy 已加载时也保留）、冻结调度/输入绑定及 next epoch/batch cursor。明确 `resume_entry_implemented=false`，没有声称已有可执行恢复入口。
14: 
15: 最长完整输入、无更新容量预检、MLP 分块重算、LoRA SUM/world 归约、nonfloat ignore、尾批真实行和零通信槽沿用旧机制。没有截断、补样、重采样或额外 update。提示/六工具/预算/评测不在本次编辑范围。
16: 
17: ## 日志口径
18: 
19: 保留旧 `mean_supervision_loss`。新增 `weighted_objective=sum(loss_weight*rowCE)/global8`，这是实际梯度目标；`mean_weighted_supervision_loss` 以实际行数作分母，尾批两者不同。每步记录 applied_learning_rate、实际消费行数、动作/说明各自的行均 CE / 有标签行数 / 正 token 数，以及 preclip norm、clip 上限/是否发生/实际缩放因子。通信空槽不计行或组件诊断。
20: 
21: ## CPU 验证
````

## s091

来源：`reports/self_c01_d03_t10_kaggle_20261115/t02_completed_observation.json:1`

````text
1: {
2:   "observed_utc": "2026-11-15T09:01:01.994150+00:00",
3:   "source": "Kaggle CLI authenticated competitions submissions API",
4:   "submission": {
5:     "ref": <OPERATION_ID>,
6:     "fileName": "submission.zip",
7:     "date": "2026-11-14T14:10:54.567000",
8:     "description": "SELF-C01-D03-T02-r01 | 31B QAT v2 | LoRA r16 joint trajectory SFT, 2 epochs | native6 | 480s/28 tools/80 turns | 8192 total/4096 thinking | T0.2 | public-task data",
9:     "status": "SubmissionStatus.COMPLETE",
````

## s092

来源：`learning_pipeline/evaluation/panel_materialize.py:43`

````text
43: def export_snapshot(source, target, model_config):
44:     import torch
45:     from safetensors.torch import load_file, save_file
46:     from export_serving import expand
47:     config=read(source/'adapter_config.json')
48:     if config['r']!=16 or config['lora_alpha']!=16 or config.get('use_rslora',False) or len(config['target_modules'])!=410:raise ValueError('Fresh r16/a16 recipe differs')
49:     native=load_file(source/'adapter_model.safetensors',device='cpu')
50:     if len(native)!=820 or any(t.dtype!=torch.float32 or not torch.isfinite(t).all() for t in native.values()):raise ValueError('Expected finite native820 FP32')
51:     output,cfg,copied=expand(native,config,read(model_config)['text_config'])
````

## s093

来源：`learning_pipeline/specs/FRESH_SFT_C03.md:38`

````text
38: 实际交付目录`dist/harness_k_a800/trajectory_cycle03_fresh_fsdp4_r01`，7件67,227,657B；source_bundle.zip67,203,586B/63成员。独立发行审查确认完整字节、真实990行服务器CPU合同、配方/原Agent/GPU/回传策略全部通过。旧C02评测运输5件207,643,257B已直接字节核同归档到`archive/harness_k_a800/trajectory_cycle02_joint_t02_t10_pair_r01_transport_D095`，IDE与服务器未动。
39: 
40: 实际31B CUDA最长检查、首更新、124步训练、加载以及候选行为评测仍NOT_RUN，须由当前zcode交接执行。中间快照只作可选权重，final未被宣称最佳；完整resume入口仍未实现。所有完成状态/报告入口在[本轮总览](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/../reports/fresh_lora_implementation_20261116/overview.md)。
41: 
42: ## 数据与监督合同
43: 
44: - 本地D04复用已通过的原生mask、动作语义及公开说明资格；重选阶段价值，避免再次审阅全部历史HTTP。每题保留必要定位、源码编辑、命中真实行为的验证、恢复/事实收尾；既有源身份不是质量分数，不能只选教师或只选成功链。
45: - 公开候选只Native/A。训练split47issue/85轨迹由两个干净Sol/max curator分24/23题阅读其全部动作/回执链；正标签reasoning必须全文已读并接受，历史未读reasoning继续masked。dev/final各9issue/8repo不用于筛选目标；B/B2不扩入首轮。
46: - 每题选主要有效路径，可保留互补真实状态。无信息重复、错误/未知源码、工具格式错、假成功不进正标签。不得把源resolved当本SDK重执行通过；保留SOURCE_NATIVE_CAPTURE_SUPPORTED与SOURCE_CONVERTED_SUPPORTED。
````

## s094

来源：`reports/checkpoint_panel_c03_20261116/overview.md:29`

````text
29: 五件压缩上传约12.5MB；source_bundle.zip 12,509,850B /65成员，字节与展开/当前源直接一致。README另按独审澄清CPU序列化fixture存在。只上传该新目录，避免再次同步训练目录。实际大小清单与最后直接绑定见delivery_receipt.json。
30: 
31: analysis回传8臂所有HTTP/ADK/patch/verification/官方trace/日志/配置/SDK路由/耗时/显存/清理/预留；中间权重、八臂精确输入ZIP保留服务器，外部引用明确NOT_RETURNED。本地可完整读行为证据，但不能声称已独立直比中间权重；选定后再取该候选准确温度的输入。--full可自含。原C03训练回传独立保留。
32: 
33: 无新hash、模型下载、依赖安装、服务器操作、教师、训练、129或Kaggle提交。旧训练发行和历史数据/adapter不变。
````

## s095

来源：`reports/checkpoint_panel_c03_20261116/overview.md:21`

````text
21: - 四卡各自worker、清理后切温；单卡启动失败不打断其他卡。CPU失败且未动GPU允许真实PRESERVED封包。未闭合子进程显式阻止切温和封包。
22: - 原env.sh的进程级PATH/LIBRARY_PATH/PYTHONNOUSERSITE/offline/proxy清理移入统一driver，避免继承训练shell造成包源漂移；官方模块不改。
23: - GPU0闲置外部≤2GiB/≤5%窄例外保留，0/1原状，2/3完成后恢复本人预留。无他人kill/reset；短IPC、现场UUID到index沿用。
24: 
25: root9项CPU检查通过；干净GPT-6.1 Sol/max对实际压缩包、八套生成配置、边界及失败路径独立16项通过，0开放阻断。真实目标GPU、模型服务和96题尚NOT_RUN，本地小张量/模拟失败不冒称目标通过。独审记录见[independent](https://github.com/fjnuslw/gemma-agent-trajectory-sft/blob/main/docs/independent/final_review.json)。最初本地tempfile ACL及独审Windows编码失败已保全；不属于评测结果。
26: 
27: ## 交付与回传
28: 
29: 五件压缩上传约12.5MB；source_bundle.zip 12,509,850B /65成员，字节与展开/当前源直接一致。README另按独审澄清CPU序列化fixture存在。只上传该新目录，避免再次同步训练目录。实际大小清单与最后直接绑定见delivery_receipt.json。
````

## s096

来源：`learning_pipeline/evaluation/lean_return.py:69`

````text
69: def pack_files(files,reference_submission,target,metadata=None,reference_requirement=None):
70:     """Original ZIP container bytes are replaced by complete member evidence."""
71:     target=Path(target)
72:     if target.exists():raise FileExistsError('Preserve the prior return')
73:     reference=Reference(reference_submission);rows=[];stored=[];saved=0;origins={}
74:     names=[safe(n) for _,n in files]
75:     if len(names)!=len(set(n.casefold() for n in names)) or any(n.startswith(PREFIX+'/') for n in names):
76:         raise ValueError('Duplicate or reserved logical path')
77:     try:
````

## s097

来源：`reports/checkpoint_panel_c03_20261116/overview.md:31`

````text
31: analysis回传8臂所有HTTP/ADK/patch/verification/官方trace/日志/配置/SDK路由/耗时/显存/清理/预留；中间权重、八臂精确输入ZIP保留服务器，外部引用明确NOT_RETURNED。本地可完整读行为证据，但不能声称已独立直比中间权重；选定后再取该候选准确温度的输入。--full可自含。原C03训练回传独立保留。
32: 
33: 无新hash、模型下载、依赖安装、服务器操作、教师、训练、129或Kaggle提交。旧训练发行和历史数据/adapter不变。
````

## s098

来源：`learning_pipeline/evaluation/panel_driver.py:246`

````text
246:         pack_files(files,ref,staging/(aid+'.tar.gz'),metadata={'arm':item['arm'],'reference_local_status':'SERVER_RETAINED_NOT_ASSUMED_LOCAL'},
247:                    reference_requirement='This panel input is retained on the server. All references were compared in full on producer; local full restoration requires fetching this exact per-arm input. All traces and non-reference bytes are present.')
248:         refs.append({'arm_id':aid,'server_path':str(ref),'bytes':ref.stat().st_size,'local_status':'NOT_RETURNED_UNLESS_FULL',
249:                      'input_state':'MATERIALIZED_INPUT' if input_ready else 'NOT_CREATED_USING_COMMON_BASE_FOR_EVIDENCE_DEDUP_ONLY'})
250:     metadata={'profile':'full' if full else 'analysis','required_external_refs':refs,'comparison':comparison,
251:               'middle_weights_policy':'Retained on server; fetch chosen input later. Never substitute final for middle weights.'}
252:     put(staging/'return_index.json',metadata)
253:     files=[(p,'arms/'+p.name) for p in sorted(staging.iterdir()) if p.is_file()]
254:     for prefix,base in [('control',CONTROL),('exports_evidence',ROOT/'work/evidence')]:
````

## s099

来源：`learning_pipeline/server/NOTICE.txt:1`

````text
1: This is a local action-SFT pilot, not a score or model-performance guarantee.
2: 
3: Task/action sources: the user's archived public development task runs for
4: FastAPI (https://github.com/fastapi/fastapi), Rich (https://github.com/Textualize/rich),
5: Requests (https://github.com/psf/requests), HTTPX (https://github.com/encode/httpx),
6: and actual offline Sol action corrections executed against the public task source.
7: The corresponding projects' licenses and notices continue to apply. No hidden
8: evaluation set, reference patch, full source snapshots or model weights are included.
9: 
````

## s100

来源：`reports/lora_method_reassessment_20261116/AYUSH_DATASET_AND_SCALE.md:7`

````text
7: 母集来自Dmitrii Gluzdov的[Gemma 4 Serving and LoRA Verification Kit](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-serving-and-lora-verification-kit)，不是Ayush另行发布的私有52条集合。官方CLI核到datasetId12312635、94个文件，元数据许可证为CC BY4.0；仓库源码和教师材料仍按随附声明保留来源。
8: 
9: 数据卡及lora_summary.csv列出：
10: 
11: | 层 | 成功轨迹 | 独立题（各层内部） | 目标回合 |
12: |---|---:|---:|---:|
13: | Native |103|55|1766|
14: | External A |14|14|382|
15: | External B |729|729|25165|
````


# Sources and attribution

This documentation release explains a project built on upstream software; it does not claim to invent those capabilities.

- Official competition SDK, tools, ADK loop and independent verifier: [Gemma 4 Developer Agent](https://www.kaggle.com/competitions/gemma-4-developer-agent).
- [PyTorch FSDP2 / fully_shard](https://docs.pytorch.org/docs/2.10/distributed.fsdp.fully_shard.html), checkpointing and SDPA.
- [Transformers Gemma4](https://github.com/huggingface/transformers/blob/v5.13.1/src/transformers/models/gemma4/modeling_gemma4.py), [PEFT LoRA](https://github.com/huggingface/peft/blob/v0.21.2/src/peft/tuners/lora/config.py), compressed-tensors, Tokenizers, Safetensors and Jinja2.
- [vLLM LoRA](https://docs.vllm.ai/en/v0.19.1/features/lora/); the project binds the competition build rather than assuming all builds with the same version are identical.
- [Scott Weeden's public training kit](https://www.kaggle.com/datasets/scottweeden/gemma-4-developer-agent-tool-lora-kit): reference for query-chunked checkpoint attention, target-only chunked CE and related memory techniques. Its published best score was not tied to the exact adapter/script used as a reference; it is not this project's result.
- [Public Serving and LoRA Verification Kit](https://www.kaggle.com/datasets/dmitriigluzdov/gemma-4-serving-and-lora-verification-kit): selected Native/A sources. Existing metadata records CC BY 4.0 for relevant metadata; repository code, model terms and demonstrations retain their respective notices. The dataset itself is not redistributed here.
- [On-Policy Expert Corrections](https://arxiv.org/abs/2512.14895): research context for student-state corrections; its results are not transferred to this project's settings. This project used offline reconstructed corrections and SFT, not online RL.

The MIT license in this repository applies to original explanatory material and newly authored synthetic teaching code. It does not relicense third-party excerpts, upstream packages, task repositories, model weights or datasets. Source excerpts are bounded and redacted; upstream copyrights and licenses continue to apply. No full third-party package source is bundled.

No proprietary teacher private thinking was collected. Public action explanations are explicitly authored training content and are distinct from private model reasoning.


## License

```text
MIT License

Copyright (c) 2026 fjnuslw

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```